In [1]:
# ============================================================
# STEP 1: MOUNT GOOGLE DRIVE AND IMPORT LIBRARIES
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import sentencepiece as spm

print("Libraries imported successfully.")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
Libraries imported successfully.
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
# ============================================================
# STEP 2: DEFINE PROJECT PATHS AND DEVICE
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/NNDL_English_Hindi_Translator"

PROCESSED_DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

TOKENIZER_DIR = os.path.join(
    PROJECT_ROOT,
    "tokenizers"
)

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "saved_models",
    "transformer_1m_improved"
)

os.makedirs(MODEL_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(
    PROCESSED_DATA_DIR,
    "train_clean_1m.parquet"
)

VALIDATION_PATH = os.path.join(
    PROCESSED_DATA_DIR,
    "validation_clean.parquet"
)

ENGLISH_TOKENIZER_PATH = os.path.join(
    TOKENIZER_DIR,
    "english_bpe_1m.model"
)

HINDI_TOKENIZER_PATH = os.path.join(
    TOKENIZER_DIR,
    "hindi_bpe_1m.model"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("Training data:", TRAIN_PATH)
print("Validation data:", VALIDATION_PATH)
print("Model folder:", MODEL_DIR)

Device: cuda
Training data: /content/drive/MyDrive/NNDL_English_Hindi_Translator/data/processed/train_clean_1m.parquet
Validation data: /content/drive/MyDrive/NNDL_English_Hindi_Translator/data/processed/validation_clean.parquet
Model folder: /content/drive/MyDrive/NNDL_English_Hindi_Translator/saved_models/transformer_1m_improved


In [3]:
# ============================================================
# STEP 3: SET RANDOM SEEDS AND LOAD TOKENIZERS
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("Random seed fixed at:", SEED)

# Load the new 1M-trained SentencePiece tokenizers
english_tokenizer = spm.SentencePieceProcessor(
    model_file=ENGLISH_TOKENIZER_PATH
)

hindi_tokenizer = spm.SentencePieceProcessor(
    model_file=HINDI_TOKENIZER_PATH
)

# Special token IDs
PAD_ID = english_tokenizer.pad_id()
UNK_ID = english_tokenizer.unk_id()
BOS_ID = english_tokenizer.bos_id()
EOS_ID = english_tokenizer.eos_id()

EN_VOCAB_SIZE = english_tokenizer.get_piece_size()
HI_VOCAB_SIZE = hindi_tokenizer.get_piece_size()

print("\nEnglish vocab size:", EN_VOCAB_SIZE)
print("Hindi vocab size:", HI_VOCAB_SIZE)

print("\nSpecial token IDs:")
print("PAD:", PAD_ID)
print("UNK:", UNK_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)

Random seed fixed at: 42

English vocab size: 16000
Hindi vocab size: 16000

Special token IDs:
PAD: 0
UNK: 1
BOS: 2
EOS: 3


In [4]:
# ============================================================
# STEP 4: LOAD TRAINING AND VALIDATION DATA
# ============================================================

train_df = pd.read_parquet(TRAIN_PATH)
validation_df = pd.read_parquet(VALIDATION_PATH)

print("Training data loaded successfully.")
print("Training shape:", train_df.shape)

print("\nValidation data loaded successfully.")
print("Validation shape:", validation_df.shape)

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

Training data loaded successfully.
Training shape: (1000000, 4)

Validation data loaded successfully.
Validation shape: (520, 2)

Training columns:
['english', 'hindi', 'english_word_count', 'hindi_word_count']

Validation columns:
['english', 'hindi']


In [5]:
# ============================================================
# STEP 5: CREATE TRANSLATION DATASET CLASS
# ============================================================

MAX_LENGTH = 64

class TranslationDataset(Dataset):
    """
    Converts English-Hindi sentence pairs into token ID sequences
    using the new 1M-trained SentencePiece tokenizers.
    """

    def __init__(self, dataframe, english_tokenizer, hindi_tokenizer):
        self.dataframe = dataframe.reset_index(drop=True)
        self.english_tokenizer = english_tokenizer
        self.hindi_tokenizer = hindi_tokenizer

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        english_text = str(
            self.dataframe.loc[index, "english"]
        )

        hindi_text = str(
            self.dataframe.loc[index, "hindi"]
        )

        # Encode English sentence
        english_ids = self.english_tokenizer.encode(
            english_text,
            out_type=int
        )

        # Encode Hindi sentence
        hindi_ids = self.hindi_tokenizer.encode(
            hindi_text,
            out_type=int
        )

        # Add BOS and EOS tokens
        english_ids = [BOS_ID] + english_ids + [EOS_ID]
        hindi_ids = [BOS_ID] + hindi_ids + [EOS_ID]

        # Truncate long sequences
        english_ids = english_ids[:MAX_LENGTH]
        hindi_ids = hindi_ids[:MAX_LENGTH]

        # Make sure EOS exists after truncation
        english_ids[-1] = EOS_ID
        hindi_ids[-1] = EOS_ID

        return {
            "src": torch.tensor(
                english_ids,
                dtype=torch.long
            ),
            "tgt": torch.tensor(
                hindi_ids,
                dtype=torch.long
            )
        }

print("TranslationDataset class created successfully.")
print("Maximum sequence length:", MAX_LENGTH)

TranslationDataset class created successfully.
Maximum sequence length: 64


In [6]:
# ============================================================
# STEP 6: CREATE COLLATE FUNCTION AND DATALOADERS
# ============================================================

from torch.nn.utils.rnn import pad_sequence

BATCH_SIZE = 64

def collate_fn(batch):
    """
    Pads variable-length source and target sequences
    inside each batch using PAD_ID.
    """

    src_batch = [item["src"] for item in batch]
    tgt_batch = [item["tgt"] for item in batch]

    src_padded = pad_sequence(
        src_batch,
        batch_first=True,
        padding_value=PAD_ID
    )

    tgt_padded = pad_sequence(
        tgt_batch,
        batch_first=True,
        padding_value=PAD_ID
    )

    return src_padded, tgt_padded


# Create dataset objects
train_dataset = TranslationDataset(
    train_df,
    english_tokenizer,
    hindi_tokenizer
)

validation_dataset = TranslationDataset(
    validation_df,
    english_tokenizer,
    hindi_tokenizer
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))

print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

Train samples: 1000000
Validation samples: 520
Train batches: 15625
Validation batches: 9


In [7]:
# ============================================================
# STEP 7: INSPECT ONE TRAINING BATCH
# ============================================================

src_batch, tgt_batch = next(iter(train_loader))

print("Source batch shape:", src_batch.shape)
print("Target batch shape:", tgt_batch.shape)

print("\nSource dtype:", src_batch.dtype)
print("Target dtype:", tgt_batch.dtype)

print("\nFirst source sequence:")
print(src_batch[0])

print("\nFirst target sequence:")
print(tgt_batch[0])

Source batch shape: torch.Size([64, 63])
Target batch shape: torch.Size([64, 64])

Source dtype: torch.int64
Target dtype: torch.int64

First source sequence:
tensor([    2,   308,   169,   503, 15621,  3487,   122,  4479,  3673,  2135,
           53,     6,  1071,    19,  2953,  1176, 15619,     3,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0])

First target sequence:
tensor([    2,   849,   954,  3653,    33,   234, 15574,   305,   788,     8,
         2117, 15570,  1078,    42,  6178,    33,  4946,   458,    11,  2119,
            3,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
       

In [8]:
# ============================================================
# STEP 8: DEFINE IMPROVED TRANSFORMER CONFIGURATION
# ============================================================

D_MODEL = 384
NUM_HEADS = 8
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
D_FF = 1536
DROPOUT = 0.1

MAX_SEQ_LENGTH = 64

print("Improved Transformer configuration")
print("----------------------------------")
print("d_model:", D_MODEL)
print("Attention heads:", NUM_HEADS)
print("Encoder layers:", NUM_ENCODER_LAYERS)
print("Decoder layers:", NUM_DECODER_LAYERS)
print("Feed-forward size:", D_FF)
print("Dropout:", DROPOUT)
print("Maximum sequence length:", MAX_SEQ_LENGTH)
print("English vocab size:", EN_VOCAB_SIZE)
print("Hindi vocab size:", HI_VOCAB_SIZE)

Improved Transformer configuration
----------------------------------
d_model: 384
Attention heads: 8
Encoder layers: 4
Decoder layers: 4
Feed-forward size: 1536
Dropout: 0.1
Maximum sequence length: 64
English vocab size: 16000
Hindi vocab size: 16000


In [9]:
# ============================================================
# STEP 9: POSITIONAL ENCODING FROM SCRATCH
# ============================================================

class PositionalEncoding(nn.Module):
    """
    Adds positional information to token embeddings
    using sine and cosine functions.
    """

    def __init__(self, d_model, max_length=64):
        super().__init__()

        # Create position indices: [0, 1, 2, ...]
        position = torch.arange(
            max_length,
            dtype=torch.float
        ).unsqueeze(1)

        # Create scaling values for sine/cosine
        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float
            ) * (-math.log(10000.0) / d_model)
        )

        # Positional encoding matrix
        pe = torch.zeros(
            max_length,
            d_model
        )

        # Even dimensions use sine
        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        # Odd dimensions use cosine
        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        # Add batch dimension
        pe = pe.unsqueeze(0)

        # Save as non-trainable buffer
        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):
        """
        x shape:
        [batch_size, sequence_length, d_model]
        """

        seq_length = x.size(1)

        return x + self.pe[:, :seq_length]


print("PositionalEncoding class created successfully.")

PositionalEncoding class created successfully.


In [10]:
# ============================================================
# STEP 10: MULTI-HEAD ATTENTION FROM SCRATCH
# ============================================================

class MultiHeadAttention(nn.Module):
    """
    Manual Multi-Head Attention implementation.

    We do NOT use nn.MultiheadAttention.
    This class performs:
    1. Linear projections for Q, K, V
    2. Split into multiple heads
    3. Scaled dot-product attention
    4. Masking
    5. Concatenate heads
    6. Final linear projection
    """

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()

        assert d_model % num_heads == 0, \
            "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Linear layers for Query, Key and Value
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # Final output projection
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        """
        Input shape:
        [batch_size, seq_len, d_model]

        Output shape:
        [batch_size, num_heads, seq_len, head_dim]
        """

        batch_size, seq_len, _ = x.size()

        x = x.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        return x.transpose(1, 2)

    def combine_heads(self, x):
        """
        Converts:
        [batch, heads, seq_len, head_dim]

        back into:
        [batch, seq_len, d_model]
        """

        batch_size, _, seq_len, _ = x.size()

        x = x.transpose(1, 2).contiguous()

        return x.view(
            batch_size,
            seq_len,
            self.d_model
        )

    def forward(
        self,
        query,
        key,
        value,
        mask=None,
        return_attention=False
    ):

        # Project Q, K and V
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))

        # ----------------------------------------------------
        # Numerically stable attention calculation
        # ----------------------------------------------------
        # Run the attention score calculation in float32.
        # This avoids NaN problems during mixed precision
        # training on the T4 GPU.
        # ----------------------------------------------------

        original_dtype = Q.dtype

        with torch.amp.autocast(
            device_type=query.device.type,
            enabled=False
        ):

            Q_float = Q.float()
            K_float = K.float()
            V_float = V.float()

            # Scaled dot-product attention
            scores = torch.matmul(
                Q_float,
                K_float.transpose(-2, -1)
            )

            scores = scores / math.sqrt(
                self.head_dim
            )

            # Apply mask when provided
            if mask is not None:
                scores = scores.masked_fill(
                    ~mask.bool(),
                    -1e9
                )

            # Convert scores to probabilities
            attention_weights = torch.softmax(
                scores,
                dim=-1
            )

            attention_weights = self.dropout(
                attention_weights
            )

            # Weighted sum of values
            context = torch.matmul(
                attention_weights,
                V_float
            )

        # Convert back to the original dtype
        context = context.to(original_dtype)

        # Combine all attention heads
        context = self.combine_heads(context)

        # Final projection
        output = self.W_o(context)

        if return_attention:
            return output, attention_weights

        return output


print("MultiHeadAttention class created successfully.")
print("No nn.MultiheadAttention used.")

MultiHeadAttention class created successfully.
No nn.MultiheadAttention used.


In [11]:
# ============================================================
# STEP 11: FEED-FORWARD NETWORK FROM SCRATCH
# ============================================================

class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network used inside
    each Transformer encoder and decoder layer.

    Flow:
    d_model -> d_ff -> ReLU -> Dropout -> d_model
    """

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()

        self.linear1 = nn.Linear(
            d_model,
            d_ff
        )

        self.linear2 = nn.Linear(
            d_ff,
            d_model
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.activation = nn.ReLU()

    def forward(self, x):

        x = self.linear1(x)

        x = self.activation(x)

        x = self.dropout(x)

        x = self.linear2(x)

        return x


print("FeedForward class created successfully.")

FeedForward class created successfully.


In [12]:
# ============================================================
# STEP 12: TRANSFORMER ENCODER LAYER FROM SCRATCH
# ============================================================

class EncoderLayer(nn.Module):
    """
    One Transformer Encoder Layer.

    Structure:
    1. Multi-Head Self-Attention
    2. Residual Connection + LayerNorm
    3. Feed-Forward Network
    4. Residual Connection + LayerNorm
    """

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        # Self-attention
        self.self_attention = MultiHeadAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout
        )

        # Feed-forward network
        self.feed_forward = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Dropout
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):

        # ----------------------------------------------------
        # 1. Multi-Head Self-Attention
        # ----------------------------------------------------
        attention_output = self.self_attention(
            query=x,
            key=x,
            value=x,
            mask=src_mask
        )

        # Residual connection + normalization
        x = self.norm1(
            x + self.dropout1(attention_output)
        )

        # ----------------------------------------------------
        # 2. Feed-Forward Network
        # ----------------------------------------------------
        ff_output = self.feed_forward(x)

        # Residual connection + normalization
        x = self.norm2(
            x + self.dropout2(ff_output)
        )

        return x


print("EncoderLayer class created successfully.")

EncoderLayer class created successfully.


In [13]:
# ============================================================
# STEP 13: TRANSFORMER DECODER LAYER FROM SCRATCH
# ============================================================

class DecoderLayer(nn.Module):
    """
    One Transformer Decoder Layer.

    Structure:
    1. Masked Multi-Head Self-Attention
    2. Residual Connection + LayerNorm
    3. Cross-Attention with Encoder Output
    4. Residual Connection + LayerNorm
    5. Feed-Forward Network
    6. Residual Connection + LayerNorm
    """

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        # Masked self-attention
        self.self_attention = MultiHeadAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout
        )

        # Cross-attention
        self.cross_attention = MultiHeadAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout
        )

        # Feed-forward network
        self.feed_forward = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        # Dropout layers
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(
        self,
        x,
        encoder_output,
        tgt_mask=None,
        src_mask=None,
        return_cross_attention=False
    ):

        # ----------------------------------------------------
        # 1. Masked Self-Attention
        # ----------------------------------------------------
        self_attention_output = self.self_attention(
            query=x,
            key=x,
            value=x,
            mask=tgt_mask
        )

        x = self.norm1(
            x + self.dropout1(self_attention_output)
        )

        # ----------------------------------------------------
        # 2. Cross-Attention
        # ----------------------------------------------------
        if return_cross_attention:

            cross_attention_output, cross_attention_weights = \
                self.cross_attention(
                    query=x,
                    key=encoder_output,
                    value=encoder_output,
                    mask=src_mask,
                    return_attention=True
                )

        else:

            cross_attention_output = self.cross_attention(
                query=x,
                key=encoder_output,
                value=encoder_output,
                mask=src_mask
            )

        x = self.norm2(
            x + self.dropout2(cross_attention_output)
        )

        # ----------------------------------------------------
        # 3. Feed-Forward Network
        # ----------------------------------------------------
        ff_output = self.feed_forward(x)

        x = self.norm3(
            x + self.dropout3(ff_output)
        )

        if return_cross_attention:
            return x, cross_attention_weights

        return x


print("DecoderLayer class created successfully.")
print("Masked self-attention and cross-attention included.")


DecoderLayer class created successfully.
Masked self-attention and cross-attention included.


In [14]:
# ============================================================
# STEP 14: COMPLETE TRANSFORMER MODEL FROM SCRATCH
# ============================================================

class TransformerFromScratch(nn.Module):
    """
    Complete English-to-Hindi Transformer built manually.

    Includes:
    - Source embedding
    - Target embedding
    - Positional encoding
    - Encoder stack
    - Decoder stack
    - Padding masks
    - Causal target mask
    - Final vocabulary projection

    Does NOT use nn.Transformer.
    """

    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model,
        num_heads,
        num_encoder_layers,
        num_decoder_layers,
        d_ff,
        dropout,
        max_length,
        pad_id
    ):
        super().__init__()

        self.d_model = d_model
        self.pad_id = pad_id

        # ----------------------------------------------------
        # Token embeddings
        # ----------------------------------------------------
        self.src_embedding = nn.Embedding(
            src_vocab_size,
            d_model,
            padding_idx=pad_id
        )

        self.tgt_embedding = nn.Embedding(
            tgt_vocab_size,
            d_model,
            padding_idx=pad_id
        )

        # ----------------------------------------------------
        # Positional encoding
        # ----------------------------------------------------
        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            max_length=max_length
        )

        self.embedding_dropout = nn.Dropout(dropout)

        # ----------------------------------------------------
        # Encoder stack
        # ----------------------------------------------------
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(
                d_model=d_model,
                num_heads=num_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(num_encoder_layers)
        ])

        # ----------------------------------------------------
        # Decoder stack
        # ----------------------------------------------------
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(
                d_model=d_model,
                num_heads=num_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(num_decoder_layers)
        ])

        # ----------------------------------------------------
        # Final output projection
        # ----------------------------------------------------
        self.output_layer = nn.Linear(
            d_model,
            tgt_vocab_size
        )

    # ========================================================
    # SOURCE PADDING MASK
    # ========================================================
    def create_src_mask(self, src):
        """
        src shape:
        [batch_size, src_len]

        Output shape:
        [batch_size, 1, 1, src_len]
        """

        return (
            src != self.pad_id
        ).unsqueeze(1).unsqueeze(2)

    # ========================================================
    # TARGET MASK
    # ========================================================
    def create_tgt_mask(self, tgt):
        """
        Combines:
        1. Target padding mask
        2. Causal mask

        Prevents the decoder from seeing future tokens.
        """

        batch_size, tgt_len = tgt.shape

        # Padding mask
        padding_mask = (
            tgt != self.pad_id
        ).unsqueeze(1).unsqueeze(2)

        # Causal mask
        causal_mask = torch.tril(
            torch.ones(
                tgt_len,
                tgt_len,
                device=tgt.device,
                dtype=torch.bool
            )
        )

        causal_mask = causal_mask.unsqueeze(0).unsqueeze(1)

        # Combine both masks
        return padding_mask & causal_mask

    # ========================================================
    # ENCODER
    # ========================================================
    def encode(self, src):

        src_mask = self.create_src_mask(src)

        src_emb = self.src_embedding(src)

        # Standard Transformer embedding scaling
        src_emb = src_emb * math.sqrt(self.d_model)

        src_emb = self.positional_encoding(src_emb)

        x = self.embedding_dropout(src_emb)

        for layer in self.encoder_layers:
            x = layer(
                x,
                src_mask=src_mask
            )

        return x, src_mask

    # ========================================================
    # DECODER
    # ========================================================
    def decode(
        self,
        tgt,
        encoder_output,
        src_mask,
        return_attention=False
    ):

        tgt_mask = self.create_tgt_mask(tgt)

        tgt_emb = self.tgt_embedding(tgt)

        tgt_emb = tgt_emb * math.sqrt(self.d_model)

        tgt_emb = self.positional_encoding(tgt_emb)

        x = self.embedding_dropout(tgt_emb)

        final_cross_attention = None

        for i, layer in enumerate(self.decoder_layers):

            # Return cross-attention only from final layer
            if (
                return_attention
                and i == len(self.decoder_layers) - 1
            ):

                x, final_cross_attention = layer(
                    x=x,
                    encoder_output=encoder_output,
                    tgt_mask=tgt_mask,
                    src_mask=src_mask,
                    return_cross_attention=True
                )

            else:

                x = layer(
                    x=x,
                    encoder_output=encoder_output,
                    tgt_mask=tgt_mask,
                    src_mask=src_mask
                )

        logits = self.output_layer(x)

        if return_attention:
            return logits, final_cross_attention

        return logits

    # ========================================================
    # COMPLETE FORWARD PASS
    # ========================================================
    def forward(
        self,
        src,
        tgt,
        return_attention=False
    ):

        encoder_output, src_mask = self.encode(src)

        return self.decode(
            tgt=tgt,
            encoder_output=encoder_output,
            src_mask=src_mask,
            return_attention=return_attention
        )


print("TransformerFromScratch class created successfully.")
print("No nn.Transformer used.")

TransformerFromScratch class created successfully.
No nn.Transformer used.


In [15]:
# ============================================================
# STEP 15: CREATE IMPROVED 1M TRANSFORMER MODEL
# ============================================================

model = TransformerFromScratch(
    src_vocab_size=EN_VOCAB_SIZE,
    tgt_vocab_size=HI_VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
    max_length=MAX_SEQ_LENGTH,
    pad_id=PAD_ID
).to(DEVICE)

# Count total parameters
total_params = sum(
    p.numel()
    for p in model.parameters()
)

# Count trainable parameters
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Improved Transformer model created successfully.")
print("Device:", DEVICE)

print("\nTotal parameters:",
      f"{total_params:,}")

print("Trainable parameters:",
      f"{trainable_params:,}")

Improved Transformer model created successfully.
Device: cuda

Total parameters: 35,012,224
Trainable parameters: 35,012,224


In [16]:
# ============================================================
# STEP 16: FORWARD-PASS SANITY CHECK
# ============================================================

model.eval()

# Take one batch
src_batch, tgt_batch = next(iter(train_loader))

src_batch = src_batch.to(DEVICE)
tgt_batch = tgt_batch.to(DEVICE)

# Decoder input excludes the final token
tgt_input = tgt_batch[:, :-1]

with torch.no_grad():
    logits = model(
        src_batch,
        tgt_input
    )

print("Source shape:", src_batch.shape)
print("Target input shape:", tgt_input.shape)
print("Logits shape:", logits.shape)

print("\nLogits finite:",
      torch.isfinite(logits).all().item())

Source shape: torch.Size([64, 50])
Target input shape: torch.Size([64, 46])
Logits shape: torch.Size([64, 46, 16000])

Logits finite: True


In [17]:
# ============================================================
# STEP 17: LOSS, OPTIMIZER AND WARMUP LR SCHEDULE
# ============================================================

# Label smoothing helps reduce overconfidence and can improve
# generalization in sequence-to-sequence translation.
LABEL_SMOOTHING = 0.1

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    label_smoothing=LABEL_SMOOTHING
)

# Adam optimizer with Transformer-style beta values
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1.0,   # actual LR will be controlled by the scheduler
    betas=(0.9, 0.98),
    eps=1e-9
)

WARMUP_STEPS = 4000

def transformer_lr(step):
    """
    Original Transformer learning-rate schedule:

    lr = d_model^(-0.5) *
         min(step^(-0.5), step * warmup^(-1.5))
    """

    step = max(step, 1)

    return (
        D_MODEL ** (-0.5)
        * min(
            step ** (-0.5),
            step * (WARMUP_STEPS ** (-1.5))
        )
    )


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=transformer_lr
)

print("Training setup created successfully.")
print("Label smoothing:", LABEL_SMOOTHING)
print("Warmup steps:", WARMUP_STEPS)

print(
    "Initial learning rate:",
    optimizer.param_groups[0]["lr"]
)

Training setup created successfully.
Label smoothing: 0.1
Warmup steps: 4000
Initial learning rate: 2.0171788261496964e-07


In [18]:
# ============================================================
# STEP 18: ONE-BATCH LOSS SANITY CHECK
# ============================================================

model.train()

# Get one batch
src_batch, tgt_batch = next(iter(train_loader))

src_batch = src_batch.to(DEVICE)
tgt_batch = tgt_batch.to(DEVICE)

# Decoder input:
# remove the final token
tgt_input = tgt_batch[:, :-1]

# Expected output:
# remove the first BOS token
tgt_output = tgt_batch[:, 1:]

# Forward pass
logits = model(
    src_batch,
    tgt_input
)

# Reshape for CrossEntropyLoss
loss = criterion(
    logits.reshape(-1, HI_VOCAB_SIZE),
    tgt_output.reshape(-1)
)

print("Logits shape:", logits.shape)
print("Target output shape:", tgt_output.shape)

print("\nLoss:", loss.item())
print("Loss finite:", torch.isfinite(loss).item())

Logits shape: torch.Size([64, 63, 16000])
Target output shape: torch.Size([64, 63])

Loss: 9.819497108459473
Loss finite: True


In [19]:
# ============================================================
# STEP 19: TRAINING FUNCTION FOR ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    criterion,
    device,
    scaler
):
    """
    Trains the Transformer for one complete epoch.

    Includes:
    - Mixed precision
    - Gradient clipping
    - Learning-rate scheduler
    - Average training loss
    """

    model.train()

    total_loss = 0.0

    for batch_index, (src, tgt) in enumerate(dataloader):

        src = src.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)

        # Decoder input starts with BOS
        # and excludes the final token
        tgt_input = tgt[:, :-1]

        # Expected output excludes BOS
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        # Mixed precision forward pass
        with torch.amp.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            logits = model(
                src,
                tgt_input
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    HI_VOCAB_SIZE
                ),
                tgt_output.reshape(-1)
            )

        # Backpropagation with gradient scaling
        scaler.scale(loss).backward()

        # Unscale gradients before clipping
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        # Update Transformer learning rate
        scheduler.step()

        total_loss += loss.item()

        # Show progress every 1000 batches
        if (batch_index + 1) % 1000 == 0:

            current_lr = optimizer.param_groups[0]["lr"]

            print(
                f"Batch {batch_index + 1}/{len(dataloader)} | "
                f"Loss: {loss.item():.4f} | "
                f"LR: {current_lr:.8f}"
            )

    average_loss = total_loss / len(dataloader)

    return average_loss


print("train_one_epoch function created successfully.")

train_one_epoch function created successfully.


In [20]:
# ============================================================
# STEP 20: VALIDATION FUNCTION
# ============================================================

def validate_model(
    model,
    dataloader,
    criterion,
    device
):
    """
    Evaluates the Transformer on the validation set.

    No gradients are calculated.
    Returns average validation loss.
    """

    model.eval()

    total_loss = 0.0

    with torch.no_grad():

        for src, tgt in dataloader:

            src = src.to(
                device,
                non_blocking=True
            )

            tgt = tgt.to(
                device,
                non_blocking=True
            )

            # Decoder input
            tgt_input = tgt[:, :-1]

            # Expected decoder output
            tgt_output = tgt[:, 1:]

            with torch.amp.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):

                logits = model(
                    src,
                    tgt_input
                )

                loss = criterion(
                    logits.reshape(
                        -1,
                        HI_VOCAB_SIZE
                    ),
                    tgt_output.reshape(-1)
                )

            total_loss += loss.item()

    average_loss = (
        total_loss / len(dataloader)
    )

    return average_loss


print("validate_model function created successfully.")

validate_model function created successfully.


In [21]:
# ============================================================
# STEP 21: MAIN TRAINING LOOP WITH CHECKPOINT SAVING
# ============================================================

# Mixed-precision gradient scaler
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

NUM_EPOCHS = 1

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "transformer_1m_improved_best.pt"
)

best_val_loss = float("inf")

training_history = []

print("Training setup ready.")
print("Epochs:", NUM_EPOCHS)
print("Best model will be saved to:")
print(BEST_MODEL_PATH)

for epoch in range(1, NUM_EPOCHS + 1):

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch}/{NUM_EPOCHS}")
    print("=" * 60)

    # -------------------------
    # Train
    # -------------------------
    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=DEVICE,
        scaler=scaler
    )

    # -------------------------
    # Validate
    # -------------------------
    val_loss = validate_model(
        model=model,
        dataloader=validation_loader,
        criterion=criterion,
        device=DEVICE
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nEpoch {epoch} completed | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"LR: {current_lr:.8f}"
    )

    # Save history
    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "learning_rate": current_lr
    })

    # Save best checkpoint
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_loss": best_val_loss,

            "config": {
                "d_model": D_MODEL,
                "num_heads": NUM_HEADS,
                "num_encoder_layers": NUM_ENCODER_LAYERS,
                "num_decoder_layers": NUM_DECODER_LAYERS,
                "d_ff": D_FF,
                "dropout": DROPOUT,
                "max_seq_length": MAX_SEQ_LENGTH,
                "src_vocab_size": EN_VOCAB_SIZE,
                "tgt_vocab_size": HI_VOCAB_SIZE,
                "pad_id": PAD_ID
            }
        }, BEST_MODEL_PATH)

        print(
            f"✓ Best checkpoint saved "
            f"(Val Loss: {best_val_loss:.4f})"
        )

Training setup ready.
Epochs: 1
Best model will be saved to:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/saved_models/transformer_1m_improved/transformer_1m_improved_best.pt

EPOCH 1/1


KeyboardInterrupt: 

In [ ]:
# ============================================================
# STEP 24: SAVE RESUME CHECKPOINT AFTER EPOCH 1
# ============================================================

RESUME_CHECKPOINT_PATH = os.path.join(
    MODEL_DIR,
    "transformer_1m_resume.pt"
)

torch.save({
    "epoch": 1,

    # Model
    "model_state_dict": model.state_dict(),

    # Training states
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "scaler_state_dict": scaler.state_dict(),

    # Best validation result
    "best_val_loss": best_val_loss,

    # Training history
    "training_history": training_history,

    # Model configuration
    "config": {
        "d_model": D_MODEL,
        "num_heads": NUM_HEADS,
        "num_encoder_layers": NUM_ENCODER_LAYERS,
        "num_decoder_layers": NUM_DECODER_LAYERS,
        "d_ff": D_FF,
        "dropout": DROPOUT,
        "max_seq_length": MAX_SEQ_LENGTH,
        "src_vocab_size": EN_VOCAB_SIZE,
        "tgt_vocab_size": HI_VOCAB_SIZE,
        "pad_id": PAD_ID
    }
}, RESUME_CHECKPOINT_PATH)

print("Resume checkpoint saved successfully.")
print("Completed epoch: 1")
print("Best validation loss:", best_val_loss)
print("Saved to:")
print(RESUME_CHECKPOINT_PATH)

Resume checkpoint saved successfully.
Completed epoch: 1
Best validation loss: 5.335837682088216
Saved to:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/saved_models/transformer_1m_improved/transformer_1m_resume.pt


In [ ]:
# ============================================================
# STEP 25: LOAD RESUME CHECKPOINT
# ============================================================

RESUME_CHECKPOINT_PATH = os.path.join(
    MODEL_DIR,
    "transformer_1m_resume.pt"
)

checkpoint = torch.load(
    RESUME_CHECKPOINT_PATH,
    map_location=DEVICE
)

# Restore model
model.load_state_dict(
    checkpoint["model_state_dict"]
)

# Restore optimizer
optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

# Restore scheduler
scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

# Restore mixed-precision scaler
scaler.load_state_dict(
    checkpoint["scaler_state_dict"]
)

# Restore training information
start_epoch = checkpoint["epoch"] + 1
best_val_loss = checkpoint["best_val_loss"]
training_history = checkpoint["training_history"]

print("Resume checkpoint loaded successfully.")
print("Training will continue from epoch:", start_epoch)
print("Best validation loss:", best_val_loss)
print("Current learning rate:",
      optimizer.param_groups[0]["lr"])

Resume checkpoint loaded successfully.
Training will continue from epoch: 2
Best validation loss: 5.335837682088216
Current learning rate: 0.00040401695763926724


In [ ]:
# ============================================================
# STEP 26: RESUME-SAFE TRAINING FROM EPOCH 2
# ============================================================

END_EPOCH = 5   # For now, train up to Epoch 5

for epoch in range(start_epoch, END_EPOCH + 1):

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch}/{END_EPOCH}")
    print("=" * 60)

    # -------------------------
    # Train
    # -------------------------
    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=DEVICE,
        scaler=scaler
    )

    # -------------------------
    # Validate
    # -------------------------
    val_loss = validate_model(
        model=model,
        dataloader=validation_loader,
        criterion=criterion,
        device=DEVICE
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nEpoch {epoch} completed | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"LR: {current_lr:.8f}"
    )

    # Save history
    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "learning_rate": current_lr
    })

    # Save best model
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_loss": best_val_loss,
            "config": {
                "d_model": D_MODEL,
                "num_heads": NUM_HEADS,
                "num_encoder_layers": NUM_ENCODER_LAYERS,
                "num_decoder_layers": NUM_DECODER_LAYERS,
                "d_ff": D_FF,
                "dropout": DROPOUT,
                "max_seq_length": MAX_SEQ_LENGTH,
                "src_vocab_size": EN_VOCAB_SIZE,
                "tgt_vocab_size": HI_VOCAB_SIZE,
                "pad_id": PAD_ID
            }
        }, BEST_MODEL_PATH)

        print(
            f"✓ Best model updated "
            f"(Val Loss: {best_val_loss:.4f})"
        )

    # --------------------------------------------------------
    # Always save resume checkpoint after every epoch
    # --------------------------------------------------------
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "training_history": training_history,
        "config": {
            "d_model": D_MODEL,
            "num_heads": NUM_HEADS,
            "num_encoder_layers": NUM_ENCODER_LAYERS,
            "num_decoder_layers": NUM_DECODER_LAYERS,
            "d_ff": D_FF,
            "dropout": DROPOUT,
            "max_seq_length": MAX_SEQ_LENGTH,
            "src_vocab_size": EN_VOCAB_SIZE,
            "tgt_vocab_size": HI_VOCAB_SIZE,
            "pad_id": PAD_ID
        }
    }, RESUME_CHECKPOINT_PATH)

    print("✓ Resume checkpoint saved after epoch", epoch)


EPOCH 5/5
Batch 1000/15625 | Loss: 4.2476 | LR: 0.00020199
Batch 2000/15625 | Loss: 4.2486 | LR: 0.00020042
Batch 3000/15625 | Loss: 4.2456 | LR: 0.00019890
Batch 4000/15625 | Loss: 4.1357 | LR: 0.00019740
Batch 5000/15625 | Loss: 4.0214 | LR: 0.00019594
Batch 6000/15625 | Loss: 4.2339 | LR: 0.00019451
Batch 7000/15625 | Loss: 4.2423 | LR: 0.00019312
Batch 8000/15625 | Loss: 4.1083 | LR: 0.00019175
Batch 9000/15625 | Loss: 4.2476 | LR: 0.00019041
Batch 10000/15625 | Loss: 4.0909 | LR: 0.00018910
Batch 11000/15625 | Loss: 4.2983 | LR: 0.00018781
Batch 12000/15625 | Loss: 4.2737 | LR: 0.00018655
Batch 13000/15625 | Loss: 4.3946 | LR: 0.00018532
Batch 14000/15625 | Loss: 4.2537 | LR: 0.00018411
Batch 15000/15625 | Loss: 3.9424 | LR: 0.00018292

Epoch 5 completed | Train Loss: 4.2549 | Val Loss: 4.8134 | LR: 0.00018219
✓ Best model updated (Val Loss: 4.8134)
✓ Resume checkpoint saved after epoch 5


In [ ]:
# ============================================================
# STEP 27: LOAD BEST TRANSFORMER CHECKPOINT
# ============================================================

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "transformer_1m_improved_best.pt"
)

best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

print("Best Transformer checkpoint loaded successfully.")
print("Best epoch:", best_checkpoint["epoch"])
print("Best validation loss:", best_checkpoint["best_val_loss"])

Best Transformer checkpoint loaded successfully.
Best epoch: 5
Best validation loss: 4.813407791985406


In [ ]:
# ============================================================
# STEP 28: GREEDY DECODING FUNCTION
# ============================================================

def greedy_translate(
    model,
    english_text,
    english_tokenizer,
    hindi_tokenizer,
    device,
    max_length=64
):
    """
    Translate one English sentence using greedy decoding.
    """

    model.eval()

    # Encode English sentence
    src_ids = english_tokenizer.encode(
        english_text,
        out_type=int
    )

    src_ids = [BOS_ID] + src_ids + [EOS_ID]

    # Truncate if needed
    src_ids = src_ids[:max_length]
    src_ids[-1] = EOS_ID

    src = torch.tensor(
        src_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        # Encode source only once
        encoder_output, src_mask = model.encode(src)

        # Start decoder with BOS
        generated = [BOS_ID]

        for _ in range(max_length - 1):

            tgt = torch.tensor(
                generated,
                dtype=torch.long
            ).unsqueeze(0).to(device)

            logits = model.decode(
                tgt=tgt,
                encoder_output=encoder_output,
                src_mask=src_mask
            )

            # Take prediction for last position
            next_token = logits[:, -1, :].argmax(
                dim=-1
            ).item()

            generated.append(next_token)

            # Stop when EOS is generated
            if next_token == EOS_ID:
                break

    # Remove BOS and EOS
    output_ids = [
        token
        for token in generated
        if token not in [BOS_ID, EOS_ID, PAD_ID]
    ]

    translation = hindi_tokenizer.decode(output_ids)

    return translation


print("Greedy decoding function created successfully.")

Greedy decoding function created successfully.


In [ ]:
print(
    greedy_translate(
        model,
        "I am happy.",
        english_tokenizer,
        hindi_tokenizer,
        DEVICE
    )
)

मुझे खुशी है।


In [ ]:
!pip install sacrebleu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 12.7 MB/s eta 0:00:00


In [ ]:
# ============================================================
# STEP 29: VALIDATION BLEU + chrF WITH GREEDY DECODING
# ============================================================

from sacrebleu.metrics import BLEU, CHRF
from tqdm.auto import tqdm

bleu_metric = BLEU()
chrf_metric = CHRF()

greedy_predictions = []
references = []

model.eval()

for _, row in tqdm(
    validation_df.iterrows(),
    total=len(validation_df),
    desc="Greedy validation"
):

    english_text = row["english"]
    hindi_reference = row["hindi"]

    prediction = greedy_translate(
        model=model,
        english_text=english_text,
        english_tokenizer=english_tokenizer,
        hindi_tokenizer=hindi_tokenizer,
        device=DEVICE,
        max_length=MAX_SEQ_LENGTH
    )

    greedy_predictions.append(prediction)
    references.append(hindi_reference)

# SacreBLEU expects references as a list of reference streams
bleu_score = bleu_metric.corpus_score(
    greedy_predictions,
    [references]
)

chrf_score = chrf_metric.corpus_score(
    greedy_predictions,
    [references]
)

print("\nGreedy Validation Results")
print("-------------------------")
print("BLEU:", round(bleu_score.score, 4))
print("chrF:", round(chrf_score.score, 4))

Greedy validation:   0%|          | 0/520 [00:00<?, ?it/s]


Greedy Validation Results
-------------------------
BLEU: 4.1432
chrF: 25.9322


In [ ]:
# ============================================================
# STEP 30: BEAM SEARCH DECODING FUNCTION
# ============================================================

def beam_translate(
    model,
    english_text,
    english_tokenizer,
    hindi_tokenizer,
    device,
    max_length=64,
    beam_size=4,
    length_penalty=0.6
):
    """
    Translate one English sentence using beam search decoding.
    """

    model.eval()

    # Encode source sentence
    src_ids = english_tokenizer.encode(
        english_text,
        out_type=int
    )

    src_ids = [BOS_ID] + src_ids + [EOS_ID]

    src_ids = src_ids[:max_length]
    src_ids[-1] = EOS_ID

    src = torch.tensor(
        src_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        # Encode source once
        encoder_output, src_mask = model.encode(src)

        # Each beam stores:
        # (generated_token_ids, cumulative_log_probability)
        beams = [
            ([BOS_ID], 0.0)
        ]

        completed = []

        for _ in range(max_length - 1):

            new_beams = []

            for tokens, score in beams:

                # If EOS already generated, keep it as completed
                if tokens[-1] == EOS_ID:
                    completed.append((tokens, score))
                    continue

                tgt = torch.tensor(
                    tokens,
                    dtype=torch.long
                ).unsqueeze(0).to(device)

                logits = model.decode(
                    tgt=tgt,
                    encoder_output=encoder_output,
                    src_mask=src_mask
                )

                # Last token prediction
                log_probs = torch.log_softmax(
                    logits[:, -1, :],
                    dim=-1
                )

                # Get top beam_size candidates
                top_log_probs, top_indices = torch.topk(
                    log_probs,
                    beam_size,
                    dim=-1
                )

                for i in range(beam_size):

                    next_token = top_indices[0, i].item()
                    next_score = top_log_probs[0, i].item()

                    new_tokens = tokens + [next_token]

                    new_score = score + next_score

                    new_beams.append(
                        (new_tokens, new_score)
                    )

            # If no active beams remain, stop
            if not new_beams:
                break

            # Rank beams using length normalization
            new_beams = sorted(
                new_beams,
                key=lambda x: (
                    x[1] /
                    (len(x[0]) ** length_penalty)
                ),
                reverse=True
            )

            # Keep only top beam_size beams
            beams = new_beams[:beam_size]

        # Add remaining beams
        completed.extend(beams)

        # Pick best final sequence
        best_tokens, best_score = max(
            completed,
            key=lambda x: (
                x[1] /
                (len(x[0]) ** length_penalty)
            )
        )

    # Remove special tokens
    output_ids = [
        token
        for token in best_tokens
        if token not in [BOS_ID, EOS_ID, PAD_ID]
    ]

    translation = hindi_tokenizer.decode(
        output_ids
    )

    return translation


print("Beam search decoding function created successfully.")

Beam search decoding function created successfully.


In [ ]:
print(
    beam_translate(
        model,
        "I am happy.",
        english_tokenizer,
        hindi_tokenizer,
        DEVICE,
        beam_size=4,
        length_penalty=0.6
    )
)

मुझे खुशी है।


In [ ]:
# ============================================================
# STEP 31: VALIDATION BLEU + chrF WITH BEAM SEARCH
# ============================================================

beam_predictions = []

model.eval()

for _, row in tqdm(
    validation_df.iterrows(),
    total=len(validation_df),
    desc="Beam validation"
):

    english_text = row["english"]

    prediction = beam_translate(
        model=model,
        english_text=english_text,
        english_tokenizer=english_tokenizer,
        hindi_tokenizer=hindi_tokenizer,
        device=DEVICE,
        max_length=MAX_SEQ_LENGTH,
        beam_size=4,
        length_penalty=0.6
    )

    beam_predictions.append(prediction)

beam_bleu = bleu_metric.corpus_score(
    beam_predictions,
    [references]
)

beam_chrf = chrf_metric.corpus_score(
    beam_predictions,
    [references]
)

print("\nBeam Search Validation Results")
print("------------------------------")
print("BLEU:", round(beam_bleu.score, 4))
print("chrF:", round(beam_chrf.score, 4))

print("\nComparison")
print("------------------------------")
print("Greedy BLEU:", round(bleu_score.score, 4))
print("Beam BLEU:", round(beam_bleu.score, 4))

print("Greedy chrF:", round(chrf_score.score, 4))
print("Beam chrF:", round(beam_chrf.score, 4))

NameError: name 'tqdm' is not defined

In [ ]:
# ============================================================
# CONTINUE TRAINING FROM EPOCH 6 TO EPOCH 8
# ============================================================

END_EPOCH = 10

for epoch in range(start_epoch, END_EPOCH + 1):

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch}/{END_EPOCH}")
    print("=" * 60)

    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=DEVICE,
        scaler=scaler
    )

    val_loss = validate_model(
        model=model,
        dataloader=validation_loader,
        criterion=criterion,
        device=DEVICE
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nEpoch {epoch} completed | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"LR: {current_lr:.8f}"
    )

    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "learning_rate": current_lr
    })

    # Save best model
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_loss": best_val_loss,
            "config": {
                "d_model": D_MODEL,
                "num_heads": NUM_HEADS,
                "num_encoder_layers": NUM_ENCODER_LAYERS,
                "num_decoder_layers": NUM_DECODER_LAYERS,
                "d_ff": D_FF,
                "dropout": DROPOUT,
                "max_seq_length": MAX_SEQ_LENGTH,
                "src_vocab_size": EN_VOCAB_SIZE,
                "tgt_vocab_size": HI_VOCAB_SIZE,
                "pad_id": PAD_ID
            }
        }, BEST_MODEL_PATH)

        print(
            f"✓ Best model updated "
            f"(Val Loss: {best_val_loss:.4f})"
        )

    # Save resume checkpoint after every epoch
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "training_history": training_history,
        "config": {
            "d_model": D_MODEL,
            "num_heads": NUM_HEADS,
            "num_encoder_layers": NUM_ENCODER_LAYERS,
            "num_decoder_layers": NUM_DECODER_LAYERS,
            "d_ff": D_FF,
            "dropout": DROPOUT,
            "max_seq_length": MAX_SEQ_LENGTH,
            "src_vocab_size": EN_VOCAB_SIZE,
            "tgt_vocab_size": HI_VOCAB_SIZE,
            "pad_id": PAD_ID
        }
    }, RESUME_CHECKPOINT_PATH)

    print("✓ Resume checkpoint saved after epoch", epoch)


EPOCH 5/10
Batch 1000/15625 | Loss: 4.3417 | LR: 0.00018092
Batch 2000/15625 | Loss: 4.1819 | LR: 0.00017979


KeyboardInterrupt: 

In [ ]:
print("start_epoch =", start_epoch)
print("END_EPOCH =", END_EPOCH)

NameError: name 'start_epoch' is not defined

In [ ]:
checkpoint = torch.load(
    RESUME_CHECKPOINT_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

scaler.load_state_dict(
    checkpoint["scaler_state_dict"]
)

start_epoch = checkpoint["epoch"] + 1
best_val_loss = checkpoint["best_val_loss"]
training_history = checkpoint["training_history"]

print("Checkpoint epoch:", checkpoint["epoch"])
print("Training will continue from epoch:", start_epoch)
print("Best validation loss:", best_val_loss)

Checkpoint epoch: 5
Training will continue from epoch: 6
Best validation loss: 4.813407791985406


In [ ]:
# ============================================================
# RELOAD LATEST RESUME CHECKPOINT
# ============================================================

checkpoint = torch.load(
    RESUME_CHECKPOINT_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

scaler.load_state_dict(
    checkpoint["scaler_state_dict"]
)

start_epoch = checkpoint["epoch"] + 1
best_val_loss = checkpoint["best_val_loss"]
training_history = checkpoint["training_history"]

print("Checkpoint epoch:", checkpoint["epoch"])
print("Training will continue from epoch:", start_epoch)
print("Best validation loss:", best_val_loss)

Checkpoint epoch: 10
Training will continue from epoch: 11
Best validation loss: 4.67630508210924


In [ ]:
# ============================================================
# LOAD BEST CHECKPOINT AFTER EPOCH 10
# ============================================================

best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

print("Best checkpoint loaded.")
print("Best epoch:", best_checkpoint["epoch"])
print("Best validation loss:", best_checkpoint["best_val_loss"])

Best checkpoint loaded.
Best epoch: 10
Best validation loss: 4.67630508210924


In [ ]:
# ============================================================
# VALIDATION BLEU + chrF WITH GREEDY DECODING
# ============================================================

greedy_predictions = []
references = []

model.eval()

for _, row in tqdm(
    validation_df.iterrows(),
    total=len(validation_df),
    desc="Greedy validation"
):

    english_text = row["english"]
    hindi_reference = row["hindi"]

    prediction = greedy_translate(
        model=model,
        english_text=english_text,
        english_tokenizer=english_tokenizer,
        hindi_tokenizer=hindi_tokenizer,
        device=DEVICE,
        max_length=MAX_SEQ_LENGTH
    )

    greedy_predictions.append(prediction)
    references.append(hindi_reference)

bleu_score = bleu_metric.corpus_score(
    greedy_predictions,
    [references]
)

chrf_score = chrf_metric.corpus_score(
    greedy_predictions,
    [references]
)

print("\nGreedy Validation Results")
print("-------------------------")
print("BLEU:", round(bleu_score.score, 4))
print("chrF:", round(chrf_score.score, 4))

Greedy validation:   0%|          | 0/520 [00:00<?, ?it/s]


Greedy Validation Results
-------------------------
BLEU: 4.7487
chrF: 27.6606


In [ ]:
# ============================================================
# RESUME SETUP
# ============================================================

# Mixed precision scaler
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "transformer_1m_improved_best.pt"
)

RESUME_CHECKPOINT_PATH = os.path.join(
    MODEL_DIR,
    "transformer_1m_resume.pt"
)

print("Resume training setup ready.")
print("Resume checkpoint exists:",
      os.path.exists(RESUME_CHECKPOINT_PATH))

Resume training setup ready.
Resume checkpoint exists: True


In [ ]:
checkpoint = torch.load(
    RESUME_CHECKPOINT_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

scaler.load_state_dict(
    checkpoint["scaler_state_dict"]
)

start_epoch = checkpoint["epoch"] + 1
best_val_loss = checkpoint["best_val_loss"]
training_history = checkpoint["training_history"]

print("Resume checkpoint loaded successfully.")
print("Training will continue from epoch:", start_epoch)
print("Best validation loss:", best_val_loss)
print(
    "Current learning rate:",
    optimizer.param_groups[0]["lr"]
)

Resume checkpoint loaded successfully.
Training will continue from epoch: 5
Best validation loss: 4.86248074637519
Current learning rate: 0.00020358900230161078


In [ ]:
# ============================================================
# STEP 22: VERIFY TRANSFORMER LEARNING-RATE SCHEDULE
# ============================================================

check_steps = [
    1,
    100,
    1000,
    4000,
    8000,
    15625
]

print("Learning-rate schedule check")
print("-" * 40)

for step in check_steps:
    lr = transformer_lr(step)

    print(
        f"Step {step:5d} -> LR: {lr:.8f}"
    )

print("\nWarmup peak occurs around step:", WARMUP_STEPS)

Learning-rate schedule check
----------------------------------------
Step     1 -> LR: 0.00000020
Step   100 -> LR: 0.00002017
Step  1000 -> LR: 0.00020172
Step  4000 -> LR: 0.00080687
Step  8000 -> LR: 0.00057054
Step 15625 -> LR: 0.00040825

Warmup peak occurs around step: 4000


In [ ]:
# ============================================================
# RESUME SETUP
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "transformer_1m_improved_best.pt"
)

RESUME_CHECKPOINT_PATH = os.path.join(
    MODEL_DIR,
    "transformer_1m_resume.pt"
)

print("Resume checkpoint exists:",
      os.path.exists(RESUME_CHECKPOINT_PATH))

Resume checkpoint exists: True


In [ ]:
# ============================================================
# RELOAD LATEST RESUME CHECKPOINT
# ============================================================

checkpoint = torch.load(
    RESUME_CHECKPOINT_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

scaler.load_state_dict(
    checkpoint["scaler_state_dict"]
)

start_epoch = checkpoint["epoch"] + 1
best_val_loss = checkpoint["best_val_loss"]
training_history = checkpoint["training_history"]

print("Checkpoint epoch:", checkpoint["epoch"])
print("Training will continue from epoch:", start_epoch)
print("Best validation loss:", best_val_loss)
print(
    "Current learning rate:",
    optimizer.param_groups[0]["lr"]
)

Checkpoint epoch: 14
Training will continue from epoch: 15
Best validation loss: 4.623740196228027
Current learning rate: 0.00010902698762773835


In [ ]:
# ============================================================
# STEP 20: VALIDATION FUNCTION
# ============================================================

def validate_model(
    model,
    dataloader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0

    with torch.no_grad():

        for src, tgt in dataloader:

            src = src.to(
                device,
                non_blocking=True
            )

            tgt = tgt.to(
                device,
                non_blocking=True
            )

            # Decoder input
            tgt_input = tgt[:, :-1]

            # Expected output
            tgt_output = tgt[:, 1:]

            with torch.amp.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):

                logits = model(
                    src,
                    tgt_input
                )

                loss = criterion(
                    logits.reshape(
                        -1,
                        HI_VOCAB_SIZE
                    ),
                    tgt_output.reshape(-1)
                )

            total_loss += loss.item()

    average_loss = (
        total_loss / len(dataloader)
    )

    return average_loss


print("validate_model function created successfully.")

validate_model function created successfully.


In [ ]:
# ============================================================
# STEP 19: TRAINING FUNCTION FOR ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    criterion,
    device,
    scaler
):

    model.train()

    total_loss = 0.0

    for batch_index, (src, tgt) in enumerate(dataloader):

        src = src.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            logits = model(
                src,
                tgt_input
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    HI_VOCAB_SIZE
                ),
                tgt_output.reshape(-1)
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        total_loss += loss.item()

        if (batch_index + 1) % 1000 == 0:

            current_lr = optimizer.param_groups[0]["lr"]

            print(
                f"Batch {batch_index + 1}/{len(dataloader)} | "
                f"Loss: {loss.item():.4f} | "
                f"LR: {current_lr:.8f}"
            )

    average_loss = total_loss / len(dataloader)

    return average_loss


print("train_one_epoch function created successfully.")

train_one_epoch function created successfully.


In [ ]:
# ============================================================
# CONTINUE TRAINING FROM EPOCH 11 TO EPOCH 20
# ============================================================

END_EPOCH = 20

print("Training will start from epoch:", start_epoch)
print("Training will end at epoch:", END_EPOCH)

for epoch in range(start_epoch, END_EPOCH + 1):

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch}/{END_EPOCH}")
    print("=" * 60)

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------
    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=DEVICE,
        scaler=scaler
    )

    # --------------------------------------------------------
    # VALIDATE
    # --------------------------------------------------------
    val_loss = validate_model(
        model=model,
        dataloader=validation_loader,
        criterion=criterion,
        device=DEVICE
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nEpoch {epoch} completed | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"LR: {current_lr:.8f}"
    )

    # --------------------------------------------------------
    # SAVE TRAINING HISTORY
    # --------------------------------------------------------
    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "learning_rate": current_lr
    })

    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save({
            "epoch": epoch,

            "model_state_dict": model.state_dict(),

            "optimizer_state_dict": optimizer.state_dict(),

            "scheduler_state_dict": scheduler.state_dict(),

            "best_val_loss": best_val_loss,

            "config": {
                "d_model": D_MODEL,
                "num_heads": NUM_HEADS,
                "num_encoder_layers": NUM_ENCODER_LAYERS,
                "num_decoder_layers": NUM_DECODER_LAYERS,
                "d_ff": D_FF,
                "dropout": DROPOUT,
                "max_seq_length": MAX_SEQ_LENGTH,
                "src_vocab_size": EN_VOCAB_SIZE,
                "tgt_vocab_size": HI_VOCAB_SIZE,
                "pad_id": PAD_ID
            }

        }, BEST_MODEL_PATH)

        print(
            f"✓ Best model updated "
            f"(Val Loss: {best_val_loss:.4f})"
        )

    # --------------------------------------------------------
    # SAVE RESUME CHECKPOINT AFTER EVERY EPOCH
    # --------------------------------------------------------
    torch.save({
        "epoch": epoch,

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "scheduler_state_dict": scheduler.state_dict(),

        "scaler_state_dict": scaler.state_dict(),

        "best_val_loss": best_val_loss,

        "training_history": training_history,

        "config": {
            "d_model": D_MODEL,
            "num_heads": NUM_HEADS,
            "num_encoder_layers": NUM_ENCODER_LAYERS,
            "num_decoder_layers": NUM_DECODER_LAYERS,
            "d_ff": D_FF,
            "dropout": DROPOUT,
            "max_seq_length": MAX_SEQ_LENGTH,
            "src_vocab_size": EN_VOCAB_SIZE,
            "tgt_vocab_size": HI_VOCAB_SIZE,
            "pad_id": PAD_ID
        }

    }, RESUME_CHECKPOINT_PATH)

    print(
        "✓ Resume checkpoint saved after epoch",
        epoch
    )

Training will start from epoch: 15
Training will end at epoch: 20

EPOCH 15/20


KeyboardInterrupt: 

In [ ]:
# ============================================================
# RECOVER AND SAVE COMPLETED EPOCH 15
# ============================================================

# Validate the model that just finished Epoch 15 training
val_loss = validate_model(
    model=model,
    dataloader=validation_loader,
    criterion=criterion,
    device=DEVICE
)

current_lr = optimizer.param_groups[0]["lr"]

print(
    f"Recovered Epoch 15 | "
    f"Val Loss: {val_loss:.4f} | "
    f"LR: {current_lr:.8f}"
)

# We don't have the exact average train loss because the loop
# stopped before returning it, so store None for train_loss.
training_history.append({
    "epoch": 15,
    "train_loss": None,
    "val_loss": val_loss,
    "learning_rate": current_lr
})

# Save best model if Epoch 15 improved validation loss
if val_loss < best_val_loss:

    best_val_loss = val_loss

    torch.save({
        "epoch": 15,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_loss": best_val_loss,
        "config": {
            "d_model": D_MODEL,
            "num_heads": NUM_HEADS,
            "num_encoder_layers": NUM_ENCODER_LAYERS,
            "num_decoder_layers": NUM_DECODER_LAYERS,
            "d_ff": D_FF,
            "dropout": DROPOUT,
            "max_seq_length": MAX_SEQ_LENGTH,
            "src_vocab_size": EN_VOCAB_SIZE,
            "tgt_vocab_size": HI_VOCAB_SIZE,
            "pad_id": PAD_ID
        }
    }, BEST_MODEL_PATH)

    print(
        f"✓ Best model updated at Epoch 15 "
        f"(Val Loss: {best_val_loss:.4f})"
    )

# Always save resume checkpoint as Epoch 15
torch.save({
    "epoch": 15,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "scaler_state_dict": scaler.state_dict(),
    "best_val_loss": best_val_loss,
    "training_history": training_history,
    "config": {
        "d_model": D_MODEL,
        "num_heads": NUM_HEADS,
        "num_encoder_layers": NUM_ENCODER_LAYERS,
        "num_decoder_layers": NUM_DECODER_LAYERS,
        "d_ff": D_FF,
        "dropout": DROPOUT,
        "max_seq_length": MAX_SEQ_LENGTH,
        "src_vocab_size": EN_VOCAB_SIZE,
        "tgt_vocab_size": HI_VOCAB_SIZE,
        "pad_id": PAD_ID
    }
}, RESUME_CHECKPOINT_PATH)

start_epoch = 16

print("✓ Resume checkpoint saved for Epoch 15")
print("Next training epoch:", start_epoch)

Recovered Epoch 15 | Val Loss: 4.6076 | LR: 0.00010531
✓ Best model updated at Epoch 15 (Val Loss: 4.6076)
✓ Resume checkpoint saved for Epoch 15
Next training epoch: 16


In [ ]:
# ============================================================
# CONTINUE TRAINING FROM EPOCH 16 TO EPOCH 20
# ============================================================

END_EPOCH = 20

print("Training will start from epoch:", start_epoch)
print("Training will end at epoch:", END_EPOCH)

for epoch in range(start_epoch, END_EPOCH + 1):

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch}/{END_EPOCH}")
    print("=" * 60)

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------
    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=DEVICE,
        scaler=scaler
    )

    # --------------------------------------------------------
    # VALIDATE
    # --------------------------------------------------------
    val_loss = validate_model(
        model=model,
        dataloader=validation_loader,
        criterion=criterion,
        device=DEVICE
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nEpoch {epoch} completed | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"LR: {current_lr:.8f}"
    )

    # --------------------------------------------------------
    # SAVE TRAINING HISTORY
    # --------------------------------------------------------
    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "learning_rate": current_lr
    })

    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_loss": best_val_loss,

            "config": {
                "d_model": D_MODEL,
                "num_heads": NUM_HEADS,
                "num_encoder_layers": NUM_ENCODER_LAYERS,
                "num_decoder_layers": NUM_DECODER_LAYERS,
                "d_ff": D_FF,
                "dropout": DROPOUT,
                "max_seq_length": MAX_SEQ_LENGTH,
                "src_vocab_size": EN_VOCAB_SIZE,
                "tgt_vocab_size": HI_VOCAB_SIZE,
                "pad_id": PAD_ID
            }

        }, BEST_MODEL_PATH)

        print(
            f"✓ Best model updated "
            f"(Val Loss: {best_val_loss:.4f})"
        )

    # --------------------------------------------------------
    # SAVE RESUME CHECKPOINT AFTER EVERY EPOCH
    # --------------------------------------------------------
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "training_history": training_history,

        "config": {
            "d_model": D_MODEL,
            "num_heads": NUM_HEADS,
            "num_encoder_layers": NUM_ENCODER_LAYERS,
            "num_decoder_layers": NUM_DECODER_LAYERS,
            "d_ff": D_FF,
            "dropout": DROPOUT,
            "max_seq_length": MAX_SEQ_LENGTH,
            "src_vocab_size": EN_VOCAB_SIZE,
            "tgt_vocab_size": HI_VOCAB_SIZE,
            "pad_id": PAD_ID
        }

    }, RESUME_CHECKPOINT_PATH)

    print(
        "✓ Resume checkpoint saved after epoch",
        epoch
    )

Training will start from epoch: 16
Training will end at epoch: 20

EPOCH 16/20
Batch 1000/15625 | Loss: 4.0494 | LR: 0.00010509
Batch 2000/15625 | Loss: 3.9084 | LR: 0.00010486
Batch 3000/15625 | Loss: 4.0149 | LR: 0.00010464
Batch 4000/15625 | Loss: 3.9332 | LR: 0.00010442
Batch 5000/15625 | Loss: 4.1364 | LR: 0.00010421
Batch 6000/15625 | Loss: 4.1066 | LR: 0.00010399
Batch 7000/15625 | Loss: 3.9121 | LR: 0.00010377
Batch 8000/15625 | Loss: 3.9396 | LR: 0.00010356
Batch 9000/15625 | Loss: 3.7880 | LR: 0.00010335
Batch 10000/15625 | Loss: 4.1362 | LR: 0.00010314
Batch 11000/15625 | Loss: 3.9686 | LR: 0.00010293
Batch 12000/15625 | Loss: 4.1523 | LR: 0.00010272
Batch 13000/15625 | Loss: 4.1641 | LR: 0.00010251
Batch 14000/15625 | Loss: 3.6182 | LR: 0.00010230
Batch 15000/15625 | Loss: 3.6678 | LR: 0.00010210

Epoch 16 completed | Train Loss: 3.9158 | Val Loss: 4.6072 | LR: 0.00010197
✓ Best model updated (Val Loss: 4.6072)
✓ Resume checkpoint saved after epoch 16

EPOCH 17/20
Batch 100

In [24]:
# ============================================================
# LOAD LATEST RESUME CHECKPOINT
# ============================================================

checkpoint = torch.load(
    RESUME_CHECKPOINT_PATH,
    map_location=DEVICE
)

# Restore model
model.load_state_dict(
    checkpoint["model_state_dict"]
)

# Restore optimizer
optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

# Restore scheduler
scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

# Restore scaler
scaler.load_state_dict(
    checkpoint["scaler_state_dict"]
)

# Restore training information
start_epoch = checkpoint["epoch"] + 1
best_val_loss = checkpoint["best_val_loss"]
training_history = checkpoint["training_history"]

print("Checkpoint epoch:", checkpoint["epoch"])
print("Training will continue from epoch:", start_epoch)
print("Best validation loss:", best_val_loss)
print(
    "Current learning rate:",
    optimizer.param_groups[0]["lr"]
)

Checkpoint epoch: 17
Training will continue from epoch: 18
Best validation loss: 4.599395275115967
Current learning rate: 9.893136121034427e-05


In [26]:
# ============================================================
# FIX GRADSCALER AFTER MOVING TO NEW COLAB RUNTIME
# ============================================================

# Remove gradients created by the failed Epoch 18 batch
optimizer.zero_grad(set_to_none=True)

# IMPORTANT:
# Use a fresh GradScaler instead of restoring the scaler state
# from the previous Colab runtime.
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

print("Fresh GradScaler created successfully.")
print("Model will continue from epoch:", start_epoch)
print(
    "Current learning rate:",
    optimizer.param_groups[0]["lr"]
)

Fresh GradScaler created successfully.
Model will continue from epoch: 18
Current learning rate: 9.893136121034427e-05


In [27]:
# ============================================================
# CONTINUE TRAINING FROM EPOCH 18 TO EPOCH 20
# ============================================================

END_EPOCH = 20

print("Training will start from epoch:", start_epoch)
print("Training will end at epoch:", END_EPOCH)

for epoch in range(start_epoch, END_EPOCH + 1):

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch}/{END_EPOCH}")
    print("=" * 60)

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------
    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=DEVICE,
        scaler=scaler
    )

    # --------------------------------------------------------
    # VALIDATE
    # --------------------------------------------------------
    val_loss = validate_model(
        model=model,
        dataloader=validation_loader,
        criterion=criterion,
        device=DEVICE
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nEpoch {epoch} completed | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"LR: {current_lr:.8f}"
    )

    # Save training history
    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "learning_rate": current_lr
    })

    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_loss": best_val_loss,
            "config": {
                "d_model": D_MODEL,
                "num_heads": NUM_HEADS,
                "num_encoder_layers": NUM_ENCODER_LAYERS,
                "num_decoder_layers": NUM_DECODER_LAYERS,
                "d_ff": D_FF,
                "dropout": DROPOUT,
                "max_seq_length": MAX_SEQ_LENGTH,
                "src_vocab_size": EN_VOCAB_SIZE,
                "tgt_vocab_size": HI_VOCAB_SIZE,
                "pad_id": PAD_ID
            }
        }, BEST_MODEL_PATH)

        print(
            f"✓ Best model updated "
            f"(Val Loss: {best_val_loss:.4f})"
        )

    # --------------------------------------------------------
    # SAVE RESUME CHECKPOINT AFTER EVERY EPOCH
    # --------------------------------------------------------
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "training_history": training_history,
        "config": {
            "d_model": D_MODEL,
            "num_heads": NUM_HEADS,
            "num_encoder_layers": NUM_ENCODER_LAYERS,
            "num_decoder_layers": NUM_DECODER_LAYERS,
            "d_ff": D_FF,
            "dropout": DROPOUT,
            "max_seq_length": MAX_SEQ_LENGTH,
            "src_vocab_size": EN_VOCAB_SIZE,
            "tgt_vocab_size": HI_VOCAB_SIZE,
            "pad_id": PAD_ID
        }
    }, RESUME_CHECKPOINT_PATH)

    print(
        "✓ Resume checkpoint saved after epoch",
        epoch
    )

Training will start from epoch: 18
Training will end at epoch: 20

EPOCH 18/20
Batch 1000/15625 | Loss: 3.6433 | LR: 0.00009875
Batch 2000/15625 | Loss: 3.9647 | LR: 0.00009856
Batch 3000/15625 | Loss: 3.8997 | LR: 0.00009838
Batch 4000/15625 | Loss: 4.1413 | LR: 0.00009820
Batch 5000/15625 | Loss: 3.8063 | LR: 0.00009801
Batch 6000/15625 | Loss: 3.7636 | LR: 0.00009783
Batch 7000/15625 | Loss: 4.1532 | LR: 0.00009766
Batch 8000/15625 | Loss: 4.0181 | LR: 0.00009748
Batch 9000/15625 | Loss: 3.9577 | LR: 0.00009730
Batch 10000/15625 | Loss: 4.0656 | LR: 0.00009712
Batch 11000/15625 | Loss: 3.7192 | LR: 0.00009695
Batch 12000/15625 | Loss: 3.8310 | LR: 0.00009677
Batch 13000/15625 | Loss: 3.8957 | LR: 0.00009660
Batch 14000/15625 | Loss: 3.8425 | LR: 0.00009643
Batch 15000/15625 | Loss: 3.7987 | LR: 0.00009626

Epoch 18 completed | Train Loss: 3.8884 | Val Loss: 4.5865 | LR: 0.00009615
✓ Best model updated (Val Loss: 4.5865)
✓ Resume checkpoint saved after epoch 18

EPOCH 19/20
Batch 100

In [28]:
# ============================================================
# LOAD BEST TRANSFORMER CHECKPOINT
# ============================================================

best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

print("Best checkpoint loaded successfully.")
print("Best epoch:", best_checkpoint["epoch"])
print("Best validation loss:", best_checkpoint["best_val_loss"])

Best checkpoint loaded successfully.
Best epoch: 19
Best validation loss: 4.574004120296902


In [30]:
!pip install sacrebleu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.9 MB/s eta 0:00:00


In [35]:
# ============================================================
# FULL GREEDY VALIDATION EVALUATION
# ============================================================

from tqdm.auto import tqdm
import sacrebleu
import torch

# Put model in evaluation mode
model.eval()

greedy_predictions = []
greedy_references = []

with torch.no_grad():

    for _, row in tqdm(
        validation_df.iterrows(),
        total=len(validation_df),
        desc="Greedy Validation"
    ):

        # ----------------------------------------------------
        # 1. GET ENGLISH INPUT + HINDI REFERENCE
        # ----------------------------------------------------
        english_text = str(row["english"])
        reference_hindi = str(row["hindi"])

        # ----------------------------------------------------
        # 2. ENCODE ENGLISH SENTENCE
        # ----------------------------------------------------
        src_ids = (
            [BOS_ID]
            + en_sp.encode(
                english_text,
                out_type=int
            )[:MAX_LENGTH - 2]
            + [EOS_ID]
        )

        src_tensor = torch.tensor(
            src_ids,
            dtype=torch.long
        ).unsqueeze(0).to(DEVICE)

        # ----------------------------------------------------
        # 3. RUN ENCODER ONCE
        # ----------------------------------------------------
        memory, src_mask = model.encode(src_tensor)

        # ----------------------------------------------------
        # 4. START DECODING WITH BOS TOKEN
        # ----------------------------------------------------
        generated = [BOS_ID]

        for _ in range(MAX_LENGTH - 1):

            tgt_tensor = torch.tensor(
                generated,
                dtype=torch.long
            ).unsqueeze(0).to(DEVICE)

            # IMPORTANT:
            # Your decode() returns only logits
            logits = model.decode(
                tgt_tensor,
                memory,
                src_mask,
                return_attention=False
            )

            # ------------------------------------------------
            # 5. GREEDY NEXT-TOKEN SELECTION
            # ------------------------------------------------
            next_token = logits[
                0, -1
            ].argmax(dim=-1).item()

            generated.append(next_token)

            # Stop when EOS is generated
            if next_token == EOS_ID:
                break

        # ----------------------------------------------------
        # 6. REMOVE SPECIAL TOKENS
        # ----------------------------------------------------
        output_ids = [
            token
            for token in generated
            if token not in [
                BOS_ID,
                EOS_ID,
                PAD_ID
            ]
        ]

        # ----------------------------------------------------
        # 7. CONVERT TOKEN IDS BACK TO HINDI TEXT
        # ----------------------------------------------------
        prediction = hi_sp.decode(output_ids)

        greedy_predictions.append(prediction)
        greedy_references.append(reference_hindi)


# ============================================================
# CALCULATE BLEU
# ============================================================

bleu = sacrebleu.corpus_bleu(
    greedy_predictions,
    [greedy_references]
)

# ============================================================
# CALCULATE chrF
# ============================================================

chrf = sacrebleu.corpus_chrf(
    greedy_predictions,
    [greedy_references]
)

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n" + "=" * 50)
print("GREEDY VALIDATION RESULTS")
print("=" * 50)

print(f"BLEU : {bleu.score:.4f}")
print(f"chrF : {chrf.score:.4f}")

# ------------------------------------------------------------
# SHOW SOME SAMPLE TRANSLATIONS
# ------------------------------------------------------------

print("\nSample Predictions:\n")

for i in range(min(5, len(greedy_predictions))):

    print("English   :", validation_df.iloc[i]["english"])
    print("Reference :", greedy_references[i])
    print("Prediction:", greedy_predictions[i])
    print("-" * 50)

Greedy Validation:   0%|          | 0/520 [00:00<?, ?it/s]


GREEDY VALIDATION RESULTS
BLEU : 4.9782
chrF : 29.4144

Sample Predictions:

English   : Students of the Dattatreya city Municipal corporation secondary school demonstrated their imagination power by creating the fictitious fort "Duttgarh".
Reference : महानगर पालिका अंतर्गत दत्तात्रय नगर माध्यमिक स्कूल के विद्यार्थियों ने काल्पनिक किला 'दत्तगढ़' बनाकर अपनी कल्पनाशक्ति का परिचय दिया।
Prediction: डॉ. डी. डी. नगर निगम ने अपने विद्यार्थियों की कल्पना शक्ति का प्रदर्शन किया है।
--------------------------------------------------
English   : With encouragement from Principal Sandhya Medpallivaar the teachers and students built the fort out of clay.
Reference : प्रधानाध्यापक संध्या मेडपल्लीवार के प्रोत्साहित करने पर शिक्षकों व विद्यार्थियों ने मिट्टïी से किले का निर्माण किया।
Prediction: सैंडल और सैंडल के छात्रों के साथ सैंडल ने सैंडल के विद्यार्थियों को मिट्टी से बाहर निकाला।
--------------------------------------------------
English   : Rajesh Gavre, the President of the MNPA teachers assoc

In [44]:
# ============================================================
# BEAM SEARCH VALIDATION
# Beam Size = 4
# Length Penalty = 0.6
# ============================================================

import torch
import torch.nn.functional as F
import sacrebleu
from tqdm.auto import tqdm

BEAM_SIZE = 4
LENGTH_PENALTY = 0.6

model.eval()


# ============================================================
# BEAM SEARCH FUNCTION
# ============================================================

def beam_search_translate(
    english_text,
    model,
    en_sp,
    hi_sp,
    device,
    beam_size=4,
    max_length=64,
    length_penalty=0.6
):

    # --------------------------------------------------------
    # 1. ENCODE ENGLISH INPUT
    # --------------------------------------------------------
    src_ids = (
        [BOS_ID]
        + en_sp.encode(
            english_text,
            out_type=int
        )[:max_length - 2]
        + [EOS_ID]
    )

    src_tensor = torch.tensor(
        src_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        # Encode English sentence once
        memory, src_mask = model.encode(src_tensor)

        # ----------------------------------------------------
        # Beam structure:
        # ([generated token IDs], cumulative log probability)
        # ----------------------------------------------------
        beams = [
            ([BOS_ID], 0.0)
        ]

        completed_beams = []

        # ----------------------------------------------------
        # 2. GENERATE TOKENS
        # ----------------------------------------------------
        for _ in range(max_length - 1):

            candidates = []

            for sequence, score in beams:

                # If already ended, move to completed list
                if sequence[-1] == EOS_ID:
                    completed_beams.append(
                        (sequence, score)
                    )
                    continue

                tgt_tensor = torch.tensor(
                    sequence,
                    dtype=torch.long
                ).unsqueeze(0).to(device)

                # Decoder returns logits
                logits = model.decode(
                    tgt_tensor,
                    memory,
                    src_mask,
                    return_attention=False
                )

                # Get probability distribution
                log_probs = F.log_softmax(
                    logits[0, -1],
                    dim=-1
                )

                # Top beam_size next tokens
                top_probs, top_ids = torch.topk(
                    log_probs,
                    beam_size
                )

                for log_prob, token_id in zip(
                    top_probs.tolist(),
                    top_ids.tolist()
                ):

                    new_sequence = (
                        sequence + [token_id]
                    )

                    new_score = (
                        score + log_prob
                    )

                    candidates.append(
                        (new_sequence, new_score)
                    )

            # ------------------------------------------------
            # If all beams completed
            # ------------------------------------------------
            if len(candidates) == 0:
                break

            # ------------------------------------------------
            # Rank candidates with length penalty
            # ------------------------------------------------
            candidates = sorted(
                candidates,
                key=lambda x: (
                    x[1] /
                    (
                        max(
                            1,
                            len(x[0]) - 1
                        )
                        ** length_penalty
                    )
                ),
                reverse=True
            )

            # Keep only best beams
            beams = candidates[:beam_size]

        # ----------------------------------------------------
        # Add unfinished beams
        # ----------------------------------------------------
        completed_beams.extend(beams)

        # ----------------------------------------------------
        # Select final best sequence
        # ----------------------------------------------------
        best_sequence, best_score = max(
            completed_beams,
            key=lambda x: (
                x[1] /
                (
                    max(
                        1,
                        len(x[0]) - 1
                    )
                    ** length_penalty
                )
            )
        )

    # --------------------------------------------------------
    # REMOVE SPECIAL TOKENS
    # --------------------------------------------------------
    output_ids = [
        token
        for token in best_sequence
        if token not in [
            BOS_ID,
            EOS_ID,
            PAD_ID
        ]
    ]

    # Convert IDs back to Hindi
    translation = hi_sp.decode(output_ids)

    return translation


# ============================================================
# RUN BEAM SEARCH ON VALIDATION SET
# ============================================================

beam_predictions = []
beam_references = []

for _, row in tqdm(
    validation_df.iterrows(),
    total=len(validation_df),
    desc="Beam Search Validation"
):

    english_text = str(
        row["english"]
    )

    reference_hindi = str(
        row["hindi"]
    )

    prediction = beam_search_translate(
        english_text=english_text,
        model=model,
        en_sp=en_sp,
        hi_sp=hi_sp,
        device=DEVICE,
        beam_size=BEAM_SIZE,
        max_length=MAX_LENGTH,
        length_penalty=LENGTH_PENALTY
    )

    beam_predictions.append(
        prediction
    )

    beam_references.append(
        reference_hindi
    )


# ============================================================
# BLEU
# ============================================================

beam_bleu = sacrebleu.corpus_bleu(
    beam_predictions,
    [beam_references]
)


# ============================================================
# chrF
# ============================================================

beam_chrf = sacrebleu.corpus_chrf(
    beam_predictions,
    [beam_references]
)


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 55)
print("BEAM SEARCH VALIDATION RESULTS")
print("=" * 55)

print(
    f"Beam Size      : {BEAM_SIZE}"
)

print(
    f"Length Penalty : {LENGTH_PENALTY}"
)

print(
    f"BLEU           : {beam_bleu.score:.4f}"
)

print(
    f"chrF           : {beam_chrf.score:.4f}"
)


# ============================================================
# COMPARE WITH GREEDY
# ============================================================

print("\nComparison:")

print(
    f"Greedy BLEU : {bleu.score:.4f}"
)

print(
    f"Beam BLEU   : {beam_bleu.score:.4f}"
)

print(
    f"Greedy chrF : {chrf.score:.4f}"
)

print(
    f"Beam chrF   : {beam_chrf.score:.4f}"
)


# ============================================================
# SAMPLE TRANSLATIONS
# ============================================================

print("\nSample Beam Search Predictions:\n")

for i in range(
    min(5, len(beam_predictions))
):

    print(
        "English   :",
        validation_df.iloc[i]["english"]
    )

    print(
        "Reference :",
        beam_references[i]
    )

    print(
        "Greedy    :",
        greedy_predictions[i]
    )

    print(
        "Beam      :",
        beam_predictions[i]
    )

    print("-" * 60)

Beam Search Validation:   0%|          | 0/520 [00:00<?, ?it/s]


BEAM SEARCH VALIDATION RESULTS
Beam Size      : 4
Length Penalty : 0.6
BLEU           : 5.9770
chrF           : 30.2137

Comparison:
Greedy BLEU : 4.9782
Beam BLEU   : 5.9770
Greedy chrF : 29.4144
Beam chrF   : 30.2137

Sample Beam Search Predictions:

English   : Students of the Dattatreya city Municipal corporation secondary school demonstrated their imagination power by creating the fictitious fort "Duttgarh".
Reference : महानगर पालिका अंतर्गत दत्तात्रय नगर माध्यमिक स्कूल के विद्यार्थियों ने काल्पनिक किला 'दत्तगढ़' बनाकर अपनी कल्पनाशक्ति का परिचय दिया।
Greedy    : डॉ. डी. डी. नगर निगम ने अपने विद्यार्थियों की कल्पना शक्ति का प्रदर्शन किया है।
Beam      : डॉ. डी. डी. नगर निगम ने अपने विद्यार्थियों की कल्पना शक्ति का प्रदर्शन किया।
------------------------------------------------------------
English   : With encouragement from Principal Sandhya Medpallivaar the teachers and students built the fort out of clay.
Reference : प्रधानाध्यापक संध्या मेडपल्लीवार के प्रोत्साहित करने पर शिक्षक

In [45]:
best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

print("Final model loaded.")
print("Epoch:", best_checkpoint["epoch"])
print("Validation loss:", best_checkpoint["best_val_loss"])

Final model loaded.
Epoch: 19
Validation loss: 4.574004120296902


In [46]:
# ============================================================
# FINAL OFFICIAL TEST SET EVALUATION
# Epoch 19 Transformer
# Beam Size = 4
# Length Penalty = 0.6
# ============================================================

import os
import pandas as pd
import torch
import torch.nn.functional as F
import sacrebleu
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. LOAD OFFICIAL TEST DATA IF NOT ALREADY IN MEMORY
# ------------------------------------------------------------

if "test_df" not in globals():

    possible_test_paths = [
        os.path.join(
            PROJECT_ROOT,
            "data",
            "processed",
            "test_clean.parquet"
        ),
        os.path.join(
            PROJECT_ROOT,
            "processed_data",
            "test_clean.parquet"
        )
    ]

    TEST_PATH = None

    for path in possible_test_paths:
        if os.path.exists(path):
            TEST_PATH = path
            break

    if TEST_PATH is None:
        raise FileNotFoundError(
            "test_clean.parquet was not found. "
            "Please check your processed test file path."
        )

    test_df = pd.read_parquet(TEST_PATH)

print("Official test samples:", len(test_df))
print(test_df.head())


# ------------------------------------------------------------
# 2. FINAL SETTINGS
# ------------------------------------------------------------

BEAM_SIZE = 4
LENGTH_PENALTY = 0.6

model.eval()

print("\nFINAL MODEL SETTINGS")
print("Epoch          :", best_checkpoint["epoch"])
print("Beam Size      :", BEAM_SIZE)
print("Length Penalty :", LENGTH_PENALTY)


# ============================================================
# 3. BEAM SEARCH FUNCTION
# ============================================================

def final_beam_translate(
    english_text,
    model,
    en_sp,
    hi_sp,
    device,
    beam_size=4,
    max_length=64,
    length_penalty=0.6
):

    # Encode English sentence
    src_ids = (
        [BOS_ID]
        + en_sp.encode(
            english_text,
            out_type=int
        )[:max_length - 2]
        + [EOS_ID]
    )

    src_tensor = torch.tensor(
        src_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        # Encoder
        memory, src_mask = model.encode(src_tensor)

        # Start with BOS token
        beams = [
            ([BOS_ID], 0.0)
        ]

        completed_beams = []

        # Generate translation
        for _ in range(max_length - 1):

            candidates = []

            for sequence, score in beams:

                # Completed sequence
                if sequence[-1] == EOS_ID:
                    completed_beams.append(
                        (sequence, score)
                    )
                    continue

                tgt_tensor = torch.tensor(
                    sequence,
                    dtype=torch.long
                ).unsqueeze(0).to(device)

                logits = model.decode(
                    tgt_tensor,
                    memory,
                    src_mask,
                    return_attention=False
                )

                log_probs = F.log_softmax(
                    logits[0, -1],
                    dim=-1
                )

                top_probs, top_ids = torch.topk(
                    log_probs,
                    beam_size
                )

                for log_prob, token_id in zip(
                    top_probs.tolist(),
                    top_ids.tolist()
                ):

                    new_sequence = (
                        sequence + [token_id]
                    )

                    new_score = (
                        score + log_prob
                    )

                    candidates.append(
                        (
                            new_sequence,
                            new_score
                        )
                    )

            if len(candidates) == 0:
                break

            # Rank beams
            candidates = sorted(
                candidates,
                key=lambda x: (
                    x[1]
                    /
                    (
                        max(
                            1,
                            len(x[0]) - 1
                        )
                        ** length_penalty
                    )
                ),
                reverse=True
            )

            beams = candidates[:beam_size]

        # Add unfinished beams
        completed_beams.extend(beams)

        # Select best translation
        best_sequence, _ = max(
            completed_beams,
            key=lambda x: (
                x[1]
                /
                (
                    max(
                        1,
                        len(x[0]) - 1
                    )
                    ** length_penalty
                )
            )
        )

    # Remove special tokens
    output_ids = [
        token
        for token in best_sequence
        if token not in [
            BOS_ID,
            EOS_ID,
            PAD_ID
        ]
    ]

    return hi_sp.decode(output_ids)


# ============================================================
# 4. TRANSLATE OFFICIAL TEST SET
# ============================================================

final_test_predictions = []
final_test_references = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="FINAL Official Test Evaluation"
):

    english_text = str(
        row["english"]
    )

    reference_hindi = str(
        row["hindi"]
    )

    prediction = final_beam_translate(
        english_text=english_text,
        model=model,
        en_sp=en_sp,
        hi_sp=hi_sp,
        device=DEVICE,
        beam_size=BEAM_SIZE,
        max_length=MAX_LENGTH,
        length_penalty=LENGTH_PENALTY
    )

    final_test_predictions.append(
        prediction
    )

    final_test_references.append(
        reference_hindi
    )


# ============================================================
# 5. CALCULATE OFFICIAL TEST BLEU
# ============================================================

final_test_bleu = sacrebleu.corpus_bleu(
    final_test_predictions,
    [final_test_references]
)


# ============================================================
# 6. CALCULATE OFFICIAL TEST chrF
# ============================================================

final_test_chrf = sacrebleu.corpus_chrf(
    final_test_predictions,
    [final_test_references]
)


# ============================================================
# 7. FINAL RESULT
# ============================================================

print("\n" + "=" * 65)
print("FINAL OFFICIAL TEST RESULTS")
print("=" * 65)

print(
    "Model          : Transformer From Scratch"
)

print(
    "Checkpoint     : Epoch",
    best_checkpoint["epoch"]
)

print(
    "Beam Size      :",
    BEAM_SIZE
)

print(
    "Length Penalty :",
    LENGTH_PENALTY
)

print(
    f"BLEU           : {final_test_bleu.score:.4f}"
)

print(
    f"chrF           : {final_test_chrf.score:.4f}"
)


# ============================================================
# 8. SHOW 5 TEST TRANSLATIONS
# ============================================================

print("\nSample Official Test Predictions:\n")

for i in range(min(5, len(test_df))):

    print(
        "English   :",
        test_df.iloc[i]["english"]
    )

    print(
        "Reference :",
        final_test_references[i]
    )

    print(
        "Prediction:",
        final_test_predictions[i]
    )

    print("-" * 70)

Official test samples: 2507
                                             english  \
0                           A black box in your car?   
1  As America's road planners struggle to find th...   
2  The devices, which track every mile a motorist...   
3  The usually dull arena of highway planning has...   
4  Libertarians have joined environmental groups ...   

                                               hindi  
0                          आपकी कार में ब्लैक बॉक्स?  
1  जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए...  
2  यह डिवाइस, जो मोटर-चालक द्वारा वाहन चलाए गए प्...  
3  आम तौर पर हाईवे नियोजन जैसा उबाऊ काम भी अचानक ...  
4  आपने द्वारा ड्राइव किए गए मील, तथा संभवतः ड्रा...  

FINAL MODEL SETTINGS
Epoch          : 19
Beam Size      : 4
Length Penalty : 0.6


FINAL Official Test Evaluation:   0%|          | 0/2507 [00:00<?, ?it/s]


FINAL OFFICIAL TEST RESULTS
Model          : Transformer From Scratch
Checkpoint     : Epoch 19
Beam Size      : 4
Length Penalty : 0.6
BLEU           : 5.8019
chrF           : 28.7418

Sample Official Test Predictions:

English   : A black box in your car?
Reference : आपकी कार में ब्लैक बॉक्स?
Prediction: अपनी गाड़ी में काले बॉक्स?
----------------------------------------------------------------------
English   : As America's road planners struggle to find the cash to mend a crumbling highway system, many are beginning to see a solution in a little black box that fits neatly by the dashboard of your car.
Reference : जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाईवे सिस्टम को सुधारने के लिए धन की कमी से जूझ रहे हैं, वहीं बहुत-से लोग इसका समाधान छोटे से ब्लैक बॉक्स में देख रहे हैं, जो आपकी कार के डैशबोर्ड पर सफ़ाई से फिट हो जाता है।
Prediction: जैसा कि अमेरिका में है, बहुत-से लोग यह देखना चाहते हैं कि आपके मार्ग पर काले रंग की काली-बिरंगे-सी बातें करने के लिए एक सड़क तंत्र की शुरुआत

In [37]:
# ============================================================
# TEST MULTIPLE BEAM SEARCH SETTINGS ON VALIDATION SET
# ============================================================

import torch
import torch.nn.functional as F
import sacrebleu
from tqdm.auto import tqdm

model.eval()

# ------------------------------------------------------------
# SETTINGS TO TEST
# ------------------------------------------------------------

beam_settings = [
    {
        "beam_size": 5,
        "length_penalty": 0.8
    },
    {
        "beam_size": 6,
        "length_penalty": 1.0
    }
]


# ============================================================
# BEAM SEARCH FUNCTION
# ============================================================

def beam_search_translate(
    english_text,
    model,
    en_sp,
    hi_sp,
    device,
    beam_size=4,
    max_length=64,
    length_penalty=0.6
):

    # Encode English sentence
    src_ids = (
        [BOS_ID]
        + en_sp.encode(
            english_text,
            out_type=int
        )[:max_length - 2]
        + [EOS_ID]
    )

    src_tensor = torch.tensor(
        src_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        # Run encoder once
        memory, src_mask = model.encode(src_tensor)

        # Each beam:
        # (generated sequence, cumulative log probability)
        beams = [
            ([BOS_ID], 0.0)
        ]

        completed_beams = []

        for _ in range(max_length - 1):

            candidates = []

            for sequence, score in beams:

                # If EOS already generated
                if sequence[-1] == EOS_ID:
                    completed_beams.append(
                        (sequence, score)
                    )
                    continue

                tgt_tensor = torch.tensor(
                    sequence,
                    dtype=torch.long
                ).unsqueeze(0).to(device)

                # Decoder output
                logits = model.decode(
                    tgt_tensor,
                    memory,
                    src_mask,
                    return_attention=False
                )

                # Convert logits to log probabilities
                log_probs = F.log_softmax(
                    logits[0, -1],
                    dim=-1
                )

                # Get best next tokens
                top_probs, top_ids = torch.topk(
                    log_probs,
                    beam_size
                )

                for log_prob, token_id in zip(
                    top_probs.tolist(),
                    top_ids.tolist()
                ):

                    new_sequence = (
                        sequence + [token_id]
                    )

                    new_score = (
                        score + log_prob
                    )

                    candidates.append(
                        (
                            new_sequence,
                            new_score
                        )
                    )

            # If no more active candidates
            if len(candidates) == 0:
                break

            # Rank using length penalty
            candidates = sorted(
                candidates,
                key=lambda x: (
                    x[1] /
                    (
                        max(
                            1,
                            len(x[0]) - 1
                        )
                        ** length_penalty
                    )
                ),
                reverse=True
            )

            # Keep only best beams
            beams = candidates[:beam_size]

        # Add remaining unfinished beams
        completed_beams.extend(beams)

        # Select best final beam
        best_sequence, best_score = max(
            completed_beams,
            key=lambda x: (
                x[1] /
                (
                    max(
                        1,
                        len(x[0]) - 1
                    )
                    ** length_penalty
                )
            )
        )

    # Remove special tokens
    output_ids = [
        token
        for token in best_sequence
        if token not in [
            BOS_ID,
            EOS_ID,
            PAD_ID
        ]
    ]

    translation = hi_sp.decode(
        output_ids
    )

    return translation


# ============================================================
# TEST EACH SETTING
# ============================================================

all_results = []

for setting in beam_settings:

    BEAM_SIZE = setting["beam_size"]
    LENGTH_PENALTY = setting["length_penalty"]

    print("\n" + "=" * 60)
    print(
        f"Testing Beam Size = {BEAM_SIZE} | "
        f"Length Penalty = {LENGTH_PENALTY}"
    )
    print("=" * 60)

    predictions = []
    references = []

    for _, row in tqdm(
        validation_df.iterrows(),
        total=len(validation_df),
        desc=f"Beam {BEAM_SIZE}, LP {LENGTH_PENALTY}"
    ):

        english_text = str(
            row["english"]
        )

        reference_hindi = str(
            row["hindi"]
        )

        prediction = beam_search_translate(
            english_text=english_text,
            model=model,
            en_sp=en_sp,
            hi_sp=hi_sp,
            device=DEVICE,
            beam_size=BEAM_SIZE,
            max_length=MAX_LENGTH,
            length_penalty=LENGTH_PENALTY
        )

        predictions.append(
            prediction
        )

        references.append(
            reference_hindi
        )

    # --------------------------------------------------------
    # BLEU
    # --------------------------------------------------------

    bleu_result = sacrebleu.corpus_bleu(
        predictions,
        [references]
    )

    # --------------------------------------------------------
    # chrF
    # --------------------------------------------------------

    chrf_result = sacrebleu.corpus_chrf(
        predictions,
        [references]
    )

    print(
        f"\nBLEU : {bleu_result.score:.4f}"
    )

    print(
        f"chrF : {chrf_result.score:.4f}"
    )

    all_results.append({
        "beam_size": BEAM_SIZE,
        "length_penalty": LENGTH_PENALTY,
        "BLEU": bleu_result.score,
        "chrF": chrf_result.score
    })


# ============================================================
# INCLUDE YOUR EXISTING BEAM-4 RESULT
# ============================================================

all_results.append({
    "beam_size": 4,
    "length_penalty": 0.6,
    "BLEU": 6.4504,
    "chrF": 30.0112
})


# ============================================================
# SORT RESULTS BY BLEU
# ============================================================

all_results = sorted(
    all_results,
    key=lambda x: x["BLEU"],
    reverse=True
)


# ============================================================
# FINAL COMPARISON
# ============================================================

print("\n" + "=" * 65)
print("FINAL BEAM SEARCH COMPARISON")
print("=" * 65)

for result in all_results:

    print(
        f"Beam={result['beam_size']} | "
        f"LP={result['length_penalty']} | "
        f"BLEU={result['BLEU']:.4f} | "
        f"chrF={result['chrF']:.4f}"
    )


# ============================================================
# BEST SETTING
# ============================================================

best_result = all_results[0]

print("\n" + "=" * 65)
print("BEST VALIDATION SETTING")
print("=" * 65)

print(
    "Beam Size      :",
    best_result["beam_size"]
)

print(
    "Length Penalty :",
    best_result["length_penalty"]
)

print(
    f"BLEU           : {best_result['BLEU']:.4f}"
)

print(
    f"chrF           : {best_result['chrF']:.4f}"
)


Testing Beam Size = 5 | Length Penalty = 0.8


Beam 5, LP 0.8:   0%|          | 0/520 [00:00<?, ?it/s]


BLEU : 6.2640
chrF : 30.1135

Testing Beam Size = 6 | Length Penalty = 1.0


Beam 6, LP 1.0:   0%|          | 0/520 [00:00<?, ?it/s]


BLEU : 6.1678
chrF : 30.4965

FINAL BEAM SEARCH COMPARISON
Beam=4 | LP=0.6 | BLEU=6.4504 | chrF=30.0112
Beam=5 | LP=0.8 | BLEU=6.2640 | chrF=30.1135
Beam=6 | LP=1.0 | BLEU=6.1678 | chrF=30.4965

BEST VALIDATION SETTING
Beam Size      : 4
Length Penalty : 0.6
BLEU           : 6.4504
chrF           : 30.0112


In [38]:
# ============================================================
# PREPARE LOW-LR FINE-TUNING FROM BEST EPOCH 19 MODEL
# ============================================================

import torch
import os

# Reload the best model checkpoint
best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.to(DEVICE)

# Smaller learning rate for fine-tuning
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-5,
    betas=(0.9, 0.98),
    eps=1e-9
)

# Fresh scaler
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

FINE_TUNE_START_EPOCH = 21
FINE_TUNE_END_EPOCH = 23

FINE_TUNE_BEST_PATH = os.path.join(
    MODEL_DIR,
    "transformer_1m_finetuned_best.pt"
)

print("Best base checkpoint loaded.")
print("Base epoch:", best_checkpoint["epoch"])
print("Base val loss:", best_checkpoint["best_val_loss"])
print("Fine-tuning LR:", optimizer.param_groups[0]["lr"])
print(
    "Fine-tuning epochs:",
    FINE_TUNE_START_EPOCH,
    "to",
    FINE_TUNE_END_EPOCH
)

Best base checkpoint loaded.
Base epoch: 19
Base val loss: 4.574004120296902
Fine-tuning LR: 2e-05
Fine-tuning epochs: 21 to 23


In [39]:
# ============================================================
# FINE-TUNING TRAIN FUNCTION
# NO SCHEDULER
# ============================================================

def fine_tune_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    device,
    scaler
):
    model.train()

    total_loss = 0.0

    for batch_idx, batch in enumerate(dataloader, start=1):

        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)

        # Decoder input excludes the last token
        tgt_input = tgt[:, :-1]

        # Target output excludes the first token
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):
            logits = model(
                src,
                tgt_input
            )

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                tgt_output.reshape(-1)
            )

        # Backpropagation
        scaler.scale(loss).backward()

        # Unscale before gradient clipping
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if batch_idx % 1000 == 0:
            print(
                f"Batch {batch_idx}/{len(dataloader)} | "
                f"Loss: {loss.item():.4f} | "
                f"LR: {optimizer.param_groups[0]['lr']:.8f}"
            )

    return total_loss / len(dataloader)

In [41]:
# ============================================================
# CORRECTED FINE-TUNING TRAIN FUNCTION
# train_loader returns (src, tgt)
# ============================================================

def fine_tune_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    device,
    scaler
):
    model.train()

    total_loss = 0.0

    for batch_idx, batch in enumerate(dataloader, start=1):

        # Your DataLoader returns a tuple
        src, tgt = batch

        src = src.to(device)
        tgt = tgt.to(device)

        # Decoder input
        tgt_input = tgt[:, :-1]

        # Expected output
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):

            logits = model(
                src,
                tgt_input
            )

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                tgt_output.reshape(-1)
            )

        # Backpropagation
        scaler.scale(loss).backward()

        # Unscale gradients before clipping
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # Optimizer step
        scaler.step(optimizer)

        # Update GradScaler
        scaler.update()

        total_loss += loss.item()

        if batch_idx % 1000 == 0:
            print(
                f"Batch {batch_idx}/{len(dataloader)} | "
                f"Loss: {loss.item():.4f} | "
                f"LR: {optimizer.param_groups[0]['lr']:.8f}"
            )

    return total_loss / len(dataloader)

In [42]:
# ============================================================
# FINE-TUNE EPOCHS 21 TO 23
# ============================================================

fine_tune_best_val = best_checkpoint["best_val_loss"]

for epoch in range(
    FINE_TUNE_START_EPOCH,
    FINE_TUNE_END_EPOCH + 1
):

    print("\n" + "=" * 60)
    print(f"FINE-TUNING EPOCH {epoch}")
    print("=" * 60)

    # -----------------------------
    # TRAIN
    # -----------------------------
    train_loss = fine_tune_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE,
        scaler=scaler
    )

    # -----------------------------
    # VALIDATE
    # -----------------------------
    val_loss = validate_model(
        model=model,
        dataloader=validation_loader,
        criterion=criterion,
        device=DEVICE
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nEpoch {epoch} completed | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"LR: {current_lr:.8f}"
    )

    # -----------------------------
    # SAVE BEST FINE-TUNED MODEL
    # -----------------------------
    if val_loss < fine_tune_best_val:

        fine_tune_best_val = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": fine_tune_best_val
        }, FINE_TUNE_BEST_PATH)

        print(
            f"✓ Fine-tuned best model updated "
            f"(Val Loss: {fine_tune_best_val:.4f})"
        )


FINE-TUNING EPOCH 21
Batch 1000/15625 | Loss: 3.5885 | LR: 0.00002000
Batch 2000/15625 | Loss: 3.8933 | LR: 0.00002000
Batch 3000/15625 | Loss: 4.0236 | LR: 0.00002000
Batch 4000/15625 | Loss: 4.1987 | LR: 0.00002000
Batch 5000/15625 | Loss: 3.7656 | LR: 0.00002000
Batch 6000/15625 | Loss: 3.6252 | LR: 0.00002000
Batch 7000/15625 | Loss: 3.7212 | LR: 0.00002000
Batch 8000/15625 | Loss: 3.5995 | LR: 0.00002000
Batch 9000/15625 | Loss: 3.8608 | LR: 0.00002000
Batch 10000/15625 | Loss: 4.1008 | LR: 0.00002000
Batch 11000/15625 | Loss: 3.8654 | LR: 0.00002000
Batch 12000/15625 | Loss: 3.8509 | LR: 0.00002000
Batch 13000/15625 | Loss: 3.6444 | LR: 0.00002000
Batch 14000/15625 | Loss: 3.5710 | LR: 0.00002000
Batch 15000/15625 | Loss: 3.9636 | LR: 0.00002000

Epoch 21 completed | Train Loss: 3.7879 | Val Loss: 4.4963 | LR: 0.00002000
✓ Fine-tuned best model updated (Val Loss: 4.4963)

FINE-TUNING EPOCH 22
Batch 1000/15625 | Loss: 3.5794 | LR: 0.00002000
Batch 2000/15625 | Loss: 3.8919 | LR: 

In [43]:
# ============================================================
# LOAD BEST FINE-TUNED MODEL
# ============================================================

fine_tuned_checkpoint = torch.load(
    FINE_TUNE_BEST_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    fine_tuned_checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

print("Fine-tuned model loaded successfully.")
print("Best fine-tuned epoch:", fine_tuned_checkpoint["epoch"])
print("Best fine-tuned val loss:", fine_tuned_checkpoint["best_val_loss"])

Fine-tuned model loaded successfully.
Best fine-tuned epoch: 23
Best fine-tuned val loss: 4.494210746553209


In [33]:
# ============================================================
# RELOAD SENTENCEPIECE TOKENIZERS
# ============================================================

import sentencepiece as spm

EN_TOKENIZER_PATH = (
    "/content/drive/MyDrive/NNDL_English_Hindi_Translator/"
    "tokenizers/english_bpe_1m.model"
)

HI_TOKENIZER_PATH = (
    "/content/drive/MyDrive/NNDL_English_Hindi_Translator/"
    "tokenizers/hindi_bpe_1m.model"
)

en_sp = spm.SentencePieceProcessor()
hi_sp = spm.SentencePieceProcessor()

en_sp.load(EN_TOKENIZER_PATH)
hi_sp.load(HI_TOKENIZER_PATH)

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

print("English tokenizer loaded:", en_sp.get_piece_size())
print("Hindi tokenizer loaded:", hi_sp.get_piece_size())

English tokenizer loaded: 16000
Hindi tokenizer loaded: 16000


In [23]:
# ============================================================
# DEFINE CHECKPOINT PATHS AGAIN
# ============================================================

import os

PROJECT_ROOT = "/content/drive/MyDrive/NNDL_English_Hindi_Translator"

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "saved_models",
    "transformer_1m_improved"
)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "transformer_1m_improved_best.pt"
)

RESUME_CHECKPOINT_PATH = os.path.join(
    MODEL_DIR,
    "transformer_1m_resume.pt"
)

print("Best model path:")
print(BEST_MODEL_PATH)

print("\nResume checkpoint path:")
print(RESUME_CHECKPOINT_PATH)

print("\nResume checkpoint exists:",
      os.path.exists(RESUME_CHECKPOINT_PATH))

Best model path:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/saved_models/transformer_1m_improved/transformer_1m_improved_best.pt

Resume checkpoint path:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/saved_models/transformer_1m_improved/transformer_1m_resume.pt

Resume checkpoint exists: True


In [ ]:
# ============================================================
# RESET GRADSCALER SAFELY
# ============================================================

# Clear any gradients left by the failed batch
optimizer.zero_grad(set_to_none=True)

# Create a fresh mixed-precision scaler
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

print("GradScaler reset successfully.")
print("Resume from epoch:", start_epoch)

GradScaler reset successfully.
Resume from epoch: 16
